# Variabilidad histórica de CO2 gaseoso en LAB004-LAB012 - auditoría sampling-repaired

**Auditoría actualizada:** LAB004/LAB007/LAB008 usan `repaired + current smoothing`; los demás LAB conservan bit a bit la señal original. La referencia forward congelada usa ahora el theta natural completo de 17 parámetros y la capa CO2 recalibrada compatible con ese theta.

**Análisis DIAGNÓSTICO retrospectivo, estrictamente descriptivo.**

Pregunta científica principal:

> ¿Cambian las conclusiones sobre variabilidad histórica, recuperación relativa de gas y posibles microfugas al reparar mejor los transientes conocidos de muestreo de LAB004, LAB007 y LAB008?

Reglas de este análisis:
- NO se recalibra nada en este notebook: se cargan parámetros ya estimados y se usan sólo en modo forward.
- Theta natural: `theta_natural_full.csv`, con sus 17 parámetros serializados explícitamente.
- Capa CO2: parámetros naturales de `co2_matrix_cross_validation_2026_full_theta_sccm_corrected/`.
- Conversión canónica: `SCCM_CORRECTED` de esa calibración (MW 44.0095, Vm 24.16 L/mol, K 0.74, 2 L).
- Solo LAB004, LAB007 y LAB008 cambian de señal observada. Para los demás LAB la igualdad con la auditoría original se verifica mediante aserciones.
- LAB009 sigue excluido de la calibración por QC y se inspecciona solo como contexto diagnóstico no canónico.
- Distinguir HECHO NUMÉRICO de INTERPRETACIÓN. Un aumento de integral no demuestra ausencia de fuga.

Commit base declarado: `9769543`. El HEAD real se registra en la celda siguiente.

Artefactos nuevos: `laboratory_2026/results/co2_historical_variability_microleaks_2026_sampling_repaired/`.


## 0. Entorno y trazabilidad

Se registra rama, HEAD y ancestroía del commit base. Fuentes canónicas: observaciones horarias QC, métricas y predicciones del experimento SCCM corregido; workbook homologado natural; `batch_summary.csv` de la calibración θ_natural; CSVs crudos de sensor (raw y filtrado) para discontinuidades.

In [ ]:
import os, json, subprocess, hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

pd.set_option('display.width', 190)
pd.set_option('display.max_columns', 60)

NB_DIR = os.getcwd()
FM = os.path.abspath(os.path.join(NB_DIR, '..', '..', '..'))
RES_OLD_MODEL = os.path.join(FM, 'laboratory_2026', 'results', 'co2_matrix_cross_validation_2026_sccm_corrected')
RES = os.path.join(FM, 'laboratory_2026', 'results', 'co2_matrix_cross_validation_2026_full_theta_sccm_corrected')
OUT_ORIGINAL = os.path.join(FM, 'laboratory_2026', 'results', 'co2_historical_variability_microleaks_2026')
OUT = os.path.join(FM, 'laboratory_2026', 'results', 'co2_historical_variability_microleaks_2026_sampling_repaired')
os.makedirs(OUT, exist_ok=True)

commit_base = '9769543'
head = subprocess.run(['git', 'rev-parse', 'HEAD'], capture_output=True, text=True, cwd=FM).stdout.strip()
branch = subprocess.run(['git', 'branch', '--show-current'], capture_output=True, text=True, cwd=FM).stdout.strip()
anc = subprocess.run(['git', 'merge-base', '--is-ancestor', commit_base, 'HEAD'], cwd=FM).returncode == 0
print('rama:', branch)
print('HEAD:', head)
print(commit_base, 'es ancestro de HEAD:', anc)

manifest = json.load(open(os.path.join(RES, 'analysis_manifest.json'), encoding='utf-8'))
conversion_manifest = manifest['sccm_conversion']
SCCM_CORRECTED = float(conversion_manifest['factor_g_l_h_per_sccm_at_2L'])
assert conversion_manifest['name'] == 'SCCM_CORRECTED'
assert np.isclose(SCCM_CORRECTED, float(conversion_manifest['required_factor']), rtol=0.0, atol=1e-15)
print('SCCM_CORRECTED [g/L/h por SCCM]:', SCCM_CORRECTED)
print('constantes:', {k: conversion_manifest[k] for k in ['MW_CO2_g_mol', 'Vm_L_mol', 'K_CO2']})

XTHIOL = os.path.join(FM, 'data', 'Laboratorio 2026', 'Vendimia_2026', 'mosto_natural_xthiol.xlsx')
BATCH_SUMMARY = os.path.join(FM, 'laboratory_2026', 'results', 'estimability_historical_natural', 'batch_summary.csv')
THETA_NATURAL_FULL = os.path.join(FM, 'laboratory_2026', 'results', 'estimability_historical_natural', 'theta_natural_full.csv')
RAW = os.path.join(FM, 'data', 'Laboratorio 2026', 'raw_data')
DESK = os.path.join(RAW, 'Desktop_CyT_Tablero_Proceso')
for p in [RES, RES_OLD_MODEL, XTHIOL, BATCH_SUMMARY, THETA_NATURAL_FULL, RAW, DESK]:
    assert os.path.exists(p), p
print('fuentes presentes')
theta_full_table = pd.read_csv(THETA_NATURAL_FULL)
expected_theta_parameters = {'mu0', 'sN', 'qN', 'qXG', 'qXF', 'betaG0', 'sG', 'betaF0', 'sF', 'qEG', 'qEF', 'iG', 'iE', 'Kd0', 'm0', 'gammaG0', 'gammaF0'}
assert len(theta_full_table) == 17 and theta_full_table['parameter'].nunique() == 17
assert set(theta_full_table['parameter']) == expected_theta_parameters
theta_validation = pd.read_csv(os.path.join(RES, 'theta_validation_summary.csv')).set_index('theta_label')
theta_sha256 = hashlib.sha256(open(THETA_NATURAL_FULL, 'rb').read()).hexdigest()
assert bool(theta_validation.loc['natural', 'all_17_present'])
assert int(theta_validation.loc['natural', 'parameter_count']) == 17
assert theta_sha256 == str(theta_validation.loc['natural', 'sha256'])
assert manifest['theta']['natural'].endswith('theta_natural_full.csv')
model_reference_audit = pd.DataFrame([{
    'theta_path': os.path.relpath(THETA_NATURAL_FULL, FM),
    'theta_parameter_count': len(theta_full_table),
    'theta_sha256': theta_sha256,
    'co2_calibration_path': os.path.relpath(RES, FM),
    'sccm_conversion': conversion_manifest['name'],
    'sccm_factor_g_l_h_per_sccm_at_2L': SCCM_CORRECTED,
    'notebook_refit_performed': False,
}])
model_reference_audit.to_csv(os.path.join(OUT, 'model_reference_audit.csv'), index=False)
print('theta natural completo verificado:', len(theta_full_table), 'parámetros | sha256:', theta_sha256)
display(theta_full_table[['parameter', 'theta', 'source']])
print('calibración CO2 congelada:', os.path.relpath(RES, FM))

LABS = ['LAB004', 'LAB005', 'LAB006', 'LAB007', 'LAB008', 'LAB010', 'LAB011', 'LAB012']
LAB_CTX = 'LAB009'  # excluido por QC; solo contexto, cadena no canónica
R_SUGAR_IDEAL = 88.02 / 180.16   # 2 CO2 / glucosa, referencia física
R_ETH_IDEAL = 44.01 / 46.07      # CO2/etanol, referencia física
ETH_G_PER_PCT = 7.8924           # % v/v -> g/L (loader)

## 1. Auditoría de datos disponibles: LAB004-LAB008 y LAB010-LAB012; LAB009 solo como contexto QC

Se cargan las observaciones horarias y las predicciones congeladas del experimento full-theta + CO2 recalibrado + SCCM corregido. Antes de continuar se construyen las tres señales reparadas y se verifica explícitamente que ningún otro LAB observado cambie. La referencia predictiva anterior se carga sólo para la comparación final.


In [ ]:
obs = pd.read_csv(os.path.join(RES, 'co2_observations_hourly.csv'))
nat = obs[obs.matrix == 'natural'].copy()
bm = pd.read_csv(os.path.join(RES, 'batch_metrics.csv'))
bn = bm[(bm.target_matrix == 'natural') & (bm.calibration_matrix == 'natural')].set_index('batch')
pr = pd.read_csv(os.path.join(RES, 'prediction_rows.csv'))
prn = pr[(pr.target_matrix == 'natural') & (pr.calibration_matrix == 'natural')].copy()
pr_old_model = pd.read_csv(os.path.join(RES_OLD_MODEL, 'prediction_rows.csv'))
prn_old_model = pr_old_model[(pr_old_model.target_matrix == 'natural') & (pr_old_model.calibration_matrix == 'natural')].copy()
assert set(nat.batch.unique()) == set(LABS)
new_keys = prn[['batch', 'time_h']].reset_index(drop=True)
old_keys = prn_old_model[['batch', 'time_h']].reset_index(drop=True)
assert new_keys.equals(old_keys)
assert np.array_equal(prn['observed_g_l_h'].to_numpy(), prn_old_model['observed_g_l_h'].to_numpy(), equal_nan=True)
print('Referencia predictiva activa: full theta natural + CO2 recalibrado; referencia anterior cargada sólo para comparación.')


In [ ]:
import sys as _repair_sys
_repair_sys.path.insert(0, FM)
from laboratory_2026 import run_co2_matrix_cross_validation_2026 as co2x
print('Sampling-repaired audit: preprocessing helper loaded read-only; no fitting.')


## LAB004 — experimental sampling-transient repair

Prueba diagnóstica limitada explícitamente a `LAB004`; no modifica el preprocesamiento global ni recalibra el modelo de CO2. Se parte de la señal horaria ya preprocesada (`co2_rate_filtered_g_l_h`) y se reutilizan los tiempos de muestreo registrados por el pipeline.

Los muestreos separados por hasta 8 h se agrupan porque sus perturbaciones pueden solaparse. Cada grupo se examina desde 1 h antes del primer muestreo hasta 8 h después del último. Solo se considera reparable si la mediana preevento está por encima de `max(0.10 g/L/h, 2 × límite de detección)`. El comienzo se fija en el primer salto que supera `max(0.05 g/L/h, 3.5 × MAD robusta de los incrementos)`. El final puede extenderse más allá de la ventana fija actual: se busca la primera región postevento de tres horas consecutivas cuyos dos incrementos sean menores que `max(0.06 g/L/h, 4 × MAD)`.

La reconstrucción afecta solamente el intervalo entre regiones limpias y usa una interpolación lineal entre las medianas de tres horas pre/post. Después se reaplica, sin cambiar parámetros, el suavizado actual del runner (mediana móvil centrada de 3 puntos + Savitzky–Golay de 5 puntos y orden 2, segmentado en los pulsos de nutrientes). Los muestreos anteriores al onset o bajo el piso instrumental quedan sin reparar si no existe soporte suficiente.

In [ ]:
# Experimento local y auditable: LAB004 solamente. No fitting, no cambios al pipeline global.
LAB_REPAIR = 'LAB004'
lab004 = nat[(nat['matrix'] == 'natural') & (nat['batch'] == LAB_REPAIR)].sort_values('t_h').reset_index(drop=True).copy()
assert LAB_REPAIR == 'LAB004' and lab004['batch'].eq('LAB004').all()

sample_schedule_lab004 = co2x.load_sampling_schedule()
sample_schedule_lab004 = sample_schedule_lab004[sample_schedule_lab004['batch'].eq(LAB_REPAIR)].sort_values('sample_time_h').copy()
sample_times = np.sort(sample_schedule_lab004['sample_time_h'].astype(float).unique())
t004 = lab004['t_h'].to_numpy(dtype=float)
q004_preprocessed = lab004['co2_rate_filtered_g_l_h'].to_numpy(dtype=float)
q004_current_smoothed = lab004['co2_rate_smoothed_g_l_h'].to_numpy(dtype=float)
detection004 = lab004['detection_limit_g_l_h'].to_numpy(dtype=float)

delta004 = np.diff(q004_preprocessed)
delta_center = float(np.nanmedian(delta004))
delta_mad = 1.4826 * float(np.nanmedian(np.abs(delta004 - delta_center)))
jump_threshold = max(0.05, 3.5 * delta_mad)
recovery_threshold = max(0.06, 4.0 * delta_mad)
active_floor = max(0.10, 2.0 * float(np.nanmedian(detection004)))

# Agrupar muestreos próximos: una apertura puede afectar varias horas y solaparse con la siguiente.
sample_groups = []
for sample_h in sample_times[(sample_times >= t004.min()) & (sample_times <= t004.max())]:
    if not sample_groups or sample_h - sample_groups[-1][-1] > 8.0:
        sample_groups.append([float(sample_h)])
    else:
        sample_groups[-1].append(float(sample_h))

q004_repaired = q004_preprocessed.copy()
replaced004 = np.zeros(len(lab004), dtype=bool)
repair_rows = []
for event_times in sample_groups:
    first_sample, last_sample = event_times[0], event_times[-1]
    pre_event = (t004 >= first_sample - 4.0) & (t004 < first_sample)
    pre_level = float(np.nanmedian(q004_preprocessed[pre_event])) if pre_event.sum() >= 3 else np.nan
    if not np.isfinite(pre_level) or pre_level < active_floor:
        continue
    search_idx = np.flatnonzero((t004 >= first_sample - 1.0) & (t004 <= last_sample + 8.0))
    shock_idx = [i for i in search_idx if i > 0 and abs(q004_preprocessed[i] - q004_preprocessed[i - 1]) >= jump_threshold]
    if not shock_idx:
        continue
    start_idx = shock_idx[0]

    recovery_idx = None
    recovery_candidates = np.flatnonzero(
        (t004 >= max(last_sample + 1.0, t004[start_idx] + 1.0)) & (t004 <= last_sample + 8.0)
    )
    for candidate in recovery_candidates:
        if candidate + 2 >= len(q004_preprocessed):
            continue
        stable_post = (
            abs(q004_preprocessed[candidate + 1] - q004_preprocessed[candidate]) <= recovery_threshold
            and abs(q004_preprocessed[candidate + 2] - q004_preprocessed[candidate + 1]) <= recovery_threshold
        )
        if stable_post:
            recovery_idx = candidate
            break
    if recovery_idx is None or recovery_idx <= start_idx:
        continue

    left_idx = start_idx - 1
    left_slice = slice(max(0, left_idx - 2), left_idx + 1)
    right_slice = slice(recovery_idx, min(len(q004_preprocessed), recovery_idx + 3))
    left_level = float(np.nanmedian(q004_preprocessed[left_slice]))
    right_level = float(np.nanmedian(q004_preprocessed[right_slice]))
    repaired_values = np.interp(
        t004[start_idx:recovery_idx], [t004[left_idx], t004[recovery_idx]], [left_level, right_level]
    )
    residual = q004_preprocessed[start_idx:recovery_idx] - repaired_values
    if not np.isfinite(residual).all() or np.max(np.abs(residual)) < jump_threshold:
        continue
    q004_repaired[start_idx:recovery_idx] = repaired_values
    replaced004[start_idx:recovery_idx] = True
    has_drop = bool(np.any(residual < -jump_threshold))
    has_spike = bool(np.any(residual > jump_threshold))
    signature = 'drop+spike' if has_drop and has_spike else ('drop' if has_drop else 'spike')
    repair_rows.append({
        'sample_times_h': ', '.join(f'{value:.0f}' for value in event_times),
        'repair_start_h': float(t004[start_idx]),
        'repair_end_h': float(t004[recovery_idx - 1]),
        'pre_clean_region_h': f'{t004[max(0, left_idx - 2)]:.0f}-{t004[left_idx]:.0f}',
        'post_clean_region_h': f'{t004[recovery_idx]:.0f}-{t004[min(len(t004) - 1, recovery_idx + 2)]:.0f}',
        'hours_replaced': int(recovery_idx - start_idx),
        'signature': signature,
        'min_residual_g_l_h': float(np.min(residual)),
        'max_residual_g_l_h': float(np.max(residual)),
    })

repair_windows_lab004 = pd.DataFrame(repair_rows)
assert not repair_windows_lab004.empty
assert int(repair_windows_lab004['hours_replaced'].sum()) == int(replaced004.sum())

# Reutilizar exactamente el suavizado vigente y comprobar que reproduce la curva actual.
pulse_lab004 = pd.DataFrame({
    'batch': [LAB_REPAIR], 'pulse_time_h': [float(bn.loc[LAB_REPAIR, 'pulse_time_h'])]
})
def apply_current_smoother_lab004(values):
    frame = lab004[['matrix', 'batch', 't_h']].copy()
    frame['co2_rate_filtered_g_l_h'] = np.asarray(values, dtype=float)
    return co2x.smooth_hourly_co2_profiles(frame, pulse_lab004)['co2_rate_smoothed_g_l_h'].to_numpy(dtype=float)

smoother_reproduction = apply_current_smoother_lab004(q004_preprocessed)
assert np.allclose(smoother_reproduction, q004_current_smoothed, rtol=0.0, atol=1e-12)
q004_repaired_smoothed = apply_current_smoother_lab004(q004_repaired)

def lab004_signal_metrics(values):
    values = np.asarray(values, dtype=float)
    onset_frame = pd.DataFrame({
        't_h': t004, 'co2_rate_g_l_h': values, 'detection_limit_g_l_h': detection004
    })
    onset_threshold = co2x._onset_threshold(onset_frame)
    peak_idx = int(np.nanargmax(values))
    return {
        'integral_total_g_l': float(np.trapz(values, t004)),
        'peak_g_l_h': float(values[peak_idx]),
        't_peak_h': float(t004[peak_idx]),
        'onset_h': float(co2x._sustained_onset_h(t004, values, onset_threshold)),
    }

metrics_before = lab004_signal_metrics(q004_current_smoothed)
metrics_after = lab004_signal_metrics(q004_repaired_smoothed)
repair_comparison_lab004 = pd.DataFrame({
    'before_current_robust_smoothed': metrics_before,
    'after_experimental_repair_smoothed': metrics_after,
})
repair_comparison_lab004['delta_after_minus_before'] = (
    repair_comparison_lab004['after_experimental_repair_smoothed']
    - repair_comparison_lab004['before_current_robust_smoothed']
)
repair_comparison_lab004['change_percent'] = (
    100.0 * repair_comparison_lab004['delta_after_minus_before']
    / repair_comparison_lab004['before_current_robust_smoothed']
)

print('LAB004 experimental detector thresholds:')
print(f'  robust MAD of hourly increments = {delta_mad:.6f} g/L/h')
print(f'  jump threshold = {jump_threshold:.6f} g/L/h')
print(f'  recovery threshold = {recovery_threshold:.6f} g/L/h')
print(f'  active-signal floor = {active_floor:.3f} g/L/h')
print(f'  total replaced hourly observations = {int(replaced004.sum())}')
display(repair_windows_lab004.round(4))
display(repair_comparison_lab004.round(5))

# Vista completa con las cuatro señales solicitadas.
fig, ax = plt.subplots(figsize=(13.5, 5.8))
ax.plot(t004, q004_preprocessed, color='#9E9E9E', lw=0.9, alpha=0.75, label='hourly preprocessed (current)')
ax.plot(t004, q004_current_smoothed, color='#E69F00', lw=1.8, ls='--', label='current robust smoothed')
ax.plot(t004, q004_repaired, color='#0072B2', lw=1.2, alpha=0.9, label='experimental repaired')
ax.plot(t004, q004_repaired_smoothed, color='#009E73', lw=2.1, label='repaired + current smoothing')
for row_i, row in repair_windows_lab004.iterrows():
    ax.axvspan(row['repair_start_h'], row['repair_end_h'], color='#D55E00', alpha=0.13,
               label='replaced interval' if row_i == 0 else None)
sample_label_used = False
for sample_h in sample_times:
    if t004.min() <= sample_h <= t004.max():
        ax.axvline(sample_h, color='#333333', lw=0.65, ls=':', alpha=0.45,
                   label='recorded sampling time' if not sample_label_used else None)
        sample_label_used = True
ax.set_xlabel('time [h]')
ax.set_ylabel('CO2 [g L$^{-1}$ h$^{-1}$]')
ax.set_title('LAB004 — experimental sampling-transient repair (no CO2 refit)')
ax.grid(alpha=0.22)
ax.legend(loc='upper right', fontsize=8, frameon=False, ncol=2)
fig.tight_layout()
plt.show()

# Zoom por ventana para auditar los intervalos sustituidos y sus regiones limpias.
n_windows = len(repair_windows_lab004)
fig, axes = plt.subplots(2, 2, figsize=(13.5, 8.5), squeeze=False)
for panel, (_, row) in zip(axes.ravel(), repair_windows_lab004.iterrows()):
    view = (t004 >= row['repair_start_h'] - 4.0) & (t004 <= row['repair_end_h'] + 4.0)
    panel.plot(t004[view], q004_preprocessed[view], color='#777777', marker='.', ms=4, lw=1.0, label='hourly preprocessed')
    panel.plot(t004[view], q004_current_smoothed[view], color='#E69F00', lw=1.6, ls='--', label='current robust smoothed')
    panel.plot(t004[view], q004_repaired[view], color='#0072B2', lw=1.5, label='experimental repaired')
    panel.plot(t004[view], q004_repaired_smoothed[view], color='#009E73', lw=2.0, label='repaired + current smoothing')
    panel.axvspan(row['repair_start_h'], row['repair_end_h'], color='#D55E00', alpha=0.13, label='replaced interval')
    row_samples = [float(value) for value in row['sample_times_h'].split(', ')]
    for sample_i, sample_h in enumerate(row_samples):
        panel.axvline(sample_h, color='#333333', lw=0.8, ls=':', alpha=0.65, label='sampling' if sample_i == 0 else None)
    panel.set_title(f"samples {row['sample_times_h']} h | repair {row['repair_start_h']:.0f}-{row['repair_end_h']:.0f} h", fontsize=10)
    panel.set_xlabel('time [h]')
    panel.set_ylabel('CO2 [g L$^{-1}$ h$^{-1}$]')
    panel.grid(alpha=0.22)
for panel in axes.ravel()[n_windows:]:
    panel.set_visible(False)
handles, labels = axes.ravel()[0].get_legend_handles_labels()
fig.suptitle('LAB004 — audit of dynamically extended repaired intervals', y=0.985, fontsize=14)
fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, 0.95), ncol=3, frameon=False, fontsize=8)
fig.tight_layout(rect=(0.0, 0.0, 1.0, 0.90), h_pad=1.5, w_pad=1.0)
plt.show()

print('Interpretation: this is a LAB004-only sensitivity experiment. The global preprocessing, model parameters,')
print('calibration results and all other LAB signals remain unchanged.')

## LAB007 — experimental sampling-transient repair

Se extiende a `LAB007` la metodología experimental de LAB004, recalculando los umbrales robustos y usando exclusivamente sus tiempos de muestreo. La selección sigue siendo local y diagnóstica: señal activa antes del evento, salto compatible con apertura/muestreo, recuperación adaptativa y tres observaciones limpias a cada lado. Una región con artefactos ya marcados por el pipeline no puede actuar como ancla limpia. No se modifica el preprocesamiento global ni se realiza fitting.

In [ ]:
# Helper experimental compartido por LAB007/LAB008; reproduce el criterio de LAB004.
def experimental_sampling_repair_for_lab(lab):
    assert lab in {'LAB007', 'LAB008'}
    group = nat[(nat['matrix'] == 'natural') & (nat['batch'] == lab)].sort_values('t_h').reset_index(drop=True).copy()
    assert not group.empty and group['batch'].eq(lab).all()
    schedule = co2x.load_sampling_schedule()
    schedule = schedule[schedule['batch'].eq(lab)].sort_values('sample_time_h').copy()
    sample_times_lab = np.sort(schedule['sample_time_h'].astype(float).unique())

    time_h = group['t_h'].to_numpy(dtype=float)
    preprocessed = group['co2_rate_filtered_g_l_h'].to_numpy(dtype=float)
    current_smoothed = group['co2_rate_smoothed_g_l_h'].to_numpy(dtype=float)
    detection_limit = group['detection_limit_g_l_h'].to_numpy(dtype=float)
    clean_current = group['artifact_fraction'].fillna(0.0).le(0.0).to_numpy(dtype=bool)

    increments = np.diff(preprocessed)
    increment_center = float(np.nanmedian(increments))
    increment_mad = 1.4826 * float(np.nanmedian(np.abs(increments - increment_center)))
    jump_threshold_lab = max(0.05, 3.5 * increment_mad)
    recovery_threshold_lab = max(0.06, 4.0 * increment_mad)
    active_floor_lab = max(0.10, 2.0 * float(np.nanmedian(detection_limit)))

    sample_groups_lab = []
    valid_samples = sample_times_lab[(sample_times_lab >= time_h.min()) & (sample_times_lab <= time_h.max())]
    for sample_h in valid_samples:
        if not sample_groups_lab or sample_h - sample_groups_lab[-1][-1] > 8.0:
            sample_groups_lab.append([float(sample_h)])
        else:
            sample_groups_lab[-1].append(float(sample_h))

    repaired = preprocessed.copy()
    replaced = np.zeros(len(group), dtype=bool)
    repair_rows_lab = []
    skipped_rows_lab = []
    for event_times in sample_groups_lab:
        first_sample, last_sample = event_times[0], event_times[-1]
        event_label = ', '.join(f'{value:.0f}' for value in event_times)
        pre_event = (time_h >= first_sample - 4.0) & (time_h < first_sample) & clean_current
        pre_level = float(np.nanmedian(preprocessed[pre_event])) if pre_event.sum() >= 3 else np.nan
        if not np.isfinite(pre_level) or pre_level < active_floor_lab:
            skipped_rows_lab.append({'sample_times_h': event_label, 'reason': 'below active floor or insufficient clean pre-event support'})
            continue

        search_idx = np.flatnonzero((time_h >= first_sample - 1.0) & (time_h <= last_sample + 8.0))
        shock_idx = [
            i for i in search_idx
            if i > 0 and abs(preprocessed[i] - preprocessed[i - 1]) >= jump_threshold_lab
        ]
        if not shock_idx:
            skipped_rows_lab.append({'sample_times_h': event_label, 'reason': 'no qualifying drop/spike'})
            continue
        start_idx = shock_idx[0]

        recovery_idx = None
        recovery_candidates = np.flatnonzero(
            (time_h >= max(last_sample + 1.0, time_h[start_idx] + 1.0))
            & (time_h <= last_sample + 8.0)
        )
        for candidate in recovery_candidates:
            if candidate + 2 >= len(preprocessed):
                continue
            clean_post = bool(clean_current[candidate:candidate + 3].all())
            stable_post = (
                abs(preprocessed[candidate + 1] - preprocessed[candidate]) <= recovery_threshold_lab
                and abs(preprocessed[candidate + 2] - preprocessed[candidate + 1]) <= recovery_threshold_lab
            )
            if clean_post and stable_post:
                recovery_idx = candidate
                break
        if recovery_idx is None or recovery_idx <= start_idx:
            skipped_rows_lab.append({'sample_times_h': event_label, 'reason': 'no clean stable post-event recovery'})
            continue

        left_idx = start_idx - 1
        left_indices = np.arange(max(0, left_idx - 2), left_idx + 1)
        right_indices = np.arange(recovery_idx, min(len(preprocessed), recovery_idx + 3))
        if (
            len(left_indices) < 3 or len(right_indices) < 3
            or not clean_current[left_indices].all() or not clean_current[right_indices].all()
        ):
            skipped_rows_lab.append({'sample_times_h': event_label, 'reason': 'pre/post interpolation anchors are not clean'})
            continue

        # Asociación obligatoria: al menos un muestreo registrado debe caer dentro de la ventana propuesta.
        associated = any(time_h[start_idx] <= value <= time_h[recovery_idx - 1] for value in event_times)
        if not associated:
            skipped_rows_lab.append({'sample_times_h': event_label, 'reason': 'shock cannot be associated directly with a recorded sample'})
            continue

        left_level = float(np.nanmedian(preprocessed[left_indices]))
        right_level = float(np.nanmedian(preprocessed[right_indices]))
        repaired_values = np.interp(
            time_h[start_idx:recovery_idx],
            [time_h[left_idx], time_h[recovery_idx]],
            [left_level, right_level],
        )
        residual = preprocessed[start_idx:recovery_idx] - repaired_values
        if not np.isfinite(residual).all() or np.max(np.abs(residual)) < jump_threshold_lab:
            skipped_rows_lab.append({'sample_times_h': event_label, 'reason': 'replacement residual below robust threshold'})
            continue

        repaired[start_idx:recovery_idx] = repaired_values
        replaced[start_idx:recovery_idx] = True
        has_drop = bool(np.any(residual < -jump_threshold_lab))
        has_spike = bool(np.any(residual > jump_threshold_lab))
        signature = 'drop+spike' if has_drop and has_spike else ('drop' if has_drop else 'spike')
        repair_rows_lab.append({
            'sample_times_h': event_label,
            'repair_start_h': float(time_h[start_idx]),
            'repair_end_h': float(time_h[recovery_idx - 1]),
            'pre_clean_region_h': f'{time_h[left_indices[0]]:.0f}-{time_h[left_indices[-1]]:.0f}',
            'post_clean_region_h': f'{time_h[right_indices[0]]:.0f}-{time_h[right_indices[-1]]:.0f}',
            'hours_replaced': int(recovery_idx - start_idx),
            'signature': signature,
            'min_residual_g_l_h': float(np.min(residual)),
            'max_residual_g_l_h': float(np.max(residual)),
        })

    windows = pd.DataFrame(repair_rows_lab)
    skipped = pd.DataFrame(skipped_rows_lab)
    assert not windows.empty
    assert int(windows['hours_replaced'].sum()) == int(replaced.sum())

    pulse = pd.DataFrame({
        'batch': [lab], 'pulse_time_h': [float(bn.loc[lab, 'pulse_time_h'])]
    })
    def apply_current_smoother(values):
        frame = group[['matrix', 'batch', 't_h']].copy()
        frame['co2_rate_filtered_g_l_h'] = np.asarray(values, dtype=float)
        return co2x.smooth_hourly_co2_profiles(frame, pulse)['co2_rate_smoothed_g_l_h'].to_numpy(dtype=float)

    reproduced = apply_current_smoother(preprocessed)
    assert np.allclose(reproduced, current_smoothed, rtol=0.0, atol=1e-12)
    repaired_smoothed = apply_current_smoother(repaired)

    def signal_metrics(values):
        values = np.asarray(values, dtype=float)
        onset_frame = pd.DataFrame({
            't_h': time_h, 'co2_rate_g_l_h': values, 'detection_limit_g_l_h': detection_limit
        })
        onset_threshold = co2x._onset_threshold(onset_frame)
        peak_idx = int(np.nanargmax(values))
        metrics = {
            'integral_total_g_l': float(np.trapz(values, time_h)),
            'peak_g_l_h': float(values[peak_idx]),
            't_peak_h': float(time_h[peak_idx]),
            'onset_h': float(co2x._sustained_onset_h(time_h, values, onset_threshold)),
        }
        return metrics, float(onset_threshold)

    metrics_before_lab, onset_threshold_before = signal_metrics(current_smoothed)
    metrics_after_lab, onset_threshold_after = signal_metrics(repaired_smoothed)
    onset_after_fixed_threshold = float(
        co2x._sustained_onset_h(time_h, repaired_smoothed, onset_threshold_before)
    )
    comparison = pd.DataFrame({
        'before_current_robust_smoothed': metrics_before_lab,
        'after_experimental_repair_smoothed': metrics_after_lab,
    })
    comparison['delta_after_minus_before'] = (
        comparison['after_experimental_repair_smoothed']
        - comparison['before_current_robust_smoothed']
    )
    comparison['change_percent'] = (
        100.0 * comparison['delta_after_minus_before']
        / comparison['before_current_robust_smoothed']
    )
    return {
        'lab': lab, 'group': group, 'time_h': time_h, 'sample_times': sample_times_lab,
        'preprocessed': preprocessed, 'current_smoothed': current_smoothed,
        'repaired': repaired, 'repaired_smoothed': repaired_smoothed, 'replaced': replaced,
        'windows': windows, 'skipped': skipped, 'comparison': comparison,
        'metrics_before': metrics_before_lab, 'metrics_after': metrics_after_lab,
        'increment_mad': increment_mad, 'jump_threshold': jump_threshold_lab,
        'recovery_threshold': recovery_threshold_lab, 'active_floor': active_floor_lab,
        'onset_threshold_before': onset_threshold_before, 'onset_threshold_after': onset_threshold_after,
        'onset_after_fixed_threshold': onset_after_fixed_threshold,
    }

def display_sampling_repair_experiment(result):
    lab = result['lab']
    time_h = result['time_h']
    windows = result['windows']
    print(f'{lab} experimental detector thresholds:')
    print(f"  robust MAD of hourly increments = {result['increment_mad']:.6f} g/L/h")
    print(f"  jump threshold = {result['jump_threshold']:.6f} g/L/h")
    print(f"  recovery threshold = {result['recovery_threshold']:.6f} g/L/h")
    print(f"  active-signal floor = {result['active_floor']:.3f} g/L/h")
    print(f"  total replaced hourly observations = {int(result['replaced'].sum())}")
    display(windows.round(4))
    display(result['comparison'].round(5))
    print(f"  onset after repair using the original onset threshold = {result['onset_after_fixed_threshold']:.5f} h")
    if not result['skipped'].empty:
        print('  sampling groups not repaired conservatively:')
        display(result['skipped'])

    fig, ax = plt.subplots(figsize=(13.5, 5.8))
    ax.plot(time_h, result['preprocessed'], color='#9E9E9E', lw=0.9, alpha=0.75, label='hourly preprocessed (current)')
    ax.plot(time_h, result['current_smoothed'], color='#E69F00', lw=1.8, ls='--', label='current robust smoothed')
    ax.plot(time_h, result['repaired'], color='#0072B2', lw=1.2, alpha=0.9, label='experimental repaired')
    ax.plot(time_h, result['repaired_smoothed'], color='#009E73', lw=2.1, label='repaired + current smoothing')
    for row_i, row in windows.iterrows():
        ax.axvspan(row['repair_start_h'], row['repair_end_h'], color='#D55E00', alpha=0.13,
                   label='replaced interval' if row_i == 0 else None)
    sample_label_used = False
    for sample_h in result['sample_times']:
        if time_h.min() <= sample_h <= time_h.max():
            ax.axvline(sample_h, color='#333333', lw=0.65, ls=':', alpha=0.45,
                       label='recorded sampling time' if not sample_label_used else None)
            sample_label_used = True
    ax.set_xlabel('time [h]')
    ax.set_ylabel('CO2 [g L$^{-1}$ h$^{-1}$]')
    ax.set_title(f'{lab} — experimental sampling-transient repair (no CO2 refit)')
    ax.grid(alpha=0.22)
    ax.legend(loc='upper right', fontsize=8, frameon=False, ncol=2)
    fig.tight_layout()
    plt.show()

    n_windows = len(windows)
    ncols = 2
    nrows = int(np.ceil(n_windows / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(13.5, 4.25 * nrows), squeeze=False)
    for panel, (_, row) in zip(axes.ravel(), windows.iterrows()):
        view = (time_h >= row['repair_start_h'] - 4.0) & (time_h <= row['repair_end_h'] + 4.0)
        panel.plot(time_h[view], result['preprocessed'][view], color='#777777', marker='.', ms=4, lw=1.0, label='hourly preprocessed')
        panel.plot(time_h[view], result['current_smoothed'][view], color='#E69F00', lw=1.6, ls='--', label='current robust smoothed')
        panel.plot(time_h[view], result['repaired'][view], color='#0072B2', lw=1.5, label='experimental repaired')
        panel.plot(time_h[view], result['repaired_smoothed'][view], color='#009E73', lw=2.0, label='repaired + current smoothing')
        panel.axvspan(row['repair_start_h'], row['repair_end_h'], color='#D55E00', alpha=0.13, label='replaced interval')
        row_samples = [float(value) for value in row['sample_times_h'].split(', ')]
        for sample_i, sample_h in enumerate(row_samples):
            panel.axvline(sample_h, color='#333333', lw=0.8, ls=':', alpha=0.65,
                          label='sampling' if sample_i == 0 else None)
        panel.set_title(
            f"samples {row['sample_times_h']} h | repair {row['repair_start_h']:.0f}-{row['repair_end_h']:.0f} h",
            fontsize=10,
        )
        panel.set_xlabel('time [h]')
        panel.set_ylabel('CO2 [g L$^{-1}$ h$^{-1}$]')
        panel.grid(alpha=0.22)
    for panel in axes.ravel()[n_windows:]:
        panel.set_visible(False)
    handles, labels = axes.ravel()[0].get_legend_handles_labels()
    fig.suptitle(f'{lab} — audit of dynamically extended repaired intervals', y=0.985, fontsize=14)
    fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, 0.95), ncol=3, frameon=False, fontsize=8)
    fig.tight_layout(rect=(0.0, 0.0, 1.0, 0.90), h_pad=1.5, w_pad=1.0)
    plt.show()

lab007_repair = experimental_sampling_repair_for_lab('LAB007')
display_sampling_repair_experiment(lab007_repair)

## LAB008 — experimental sampling-transient repair

Aplicación independiente de la misma metodología a `LAB008`, usando únicamente su señal, su escala robusta y sus muestreos registrados. Las ventanas y su duración se vuelven a detectar; no se copian desde LAB004 ni LAB007.

In [ ]:
lab008_repair = experimental_sampling_repair_for_lab('LAB008')
display_sampling_repair_experiment(lab008_repair)

### Comparación compacta LAB004 / LAB007 / LAB008

Las métricas comparan en todos los casos la curva `robust smoothed` actual contra la señal experimental reparada y sometida al mismo suavizado. El onset se recalcula con la definición canónica, cuyo umbral depende del peak global; por eso también se informa en cada sección el onset reparado evaluado con el umbral original.

In [ ]:
comparison_rows = [{
    'LAB': 'LAB004',
    'integral_before_g_l': metrics_before['integral_total_g_l'],
    'integral_after_g_l': metrics_after['integral_total_g_l'],
    'integral_change_pct': 100.0 * (metrics_after['integral_total_g_l'] - metrics_before['integral_total_g_l']) / metrics_before['integral_total_g_l'],
    'peak_before_g_l_h': metrics_before['peak_g_l_h'],
    'peak_after_g_l_h': metrics_after['peak_g_l_h'],
    't_peak_before_h': metrics_before['t_peak_h'],
    't_peak_after_h': metrics_after['t_peak_h'],
    'onset_before_h': metrics_before['onset_h'],
    'onset_after_h': metrics_after['onset_h'],
    'hours_repaired': int(replaced004.sum()),
}]
for result in (lab007_repair, lab008_repair):
    before = result['metrics_before']
    after = result['metrics_after']
    comparison_rows.append({
        'LAB': result['lab'],
        'integral_before_g_l': before['integral_total_g_l'],
        'integral_after_g_l': after['integral_total_g_l'],
        'integral_change_pct': 100.0 * (after['integral_total_g_l'] - before['integral_total_g_l']) / before['integral_total_g_l'],
        'peak_before_g_l_h': before['peak_g_l_h'],
        'peak_after_g_l_h': after['peak_g_l_h'],
        't_peak_before_h': before['t_peak_h'],
        't_peak_after_h': after['t_peak_h'],
        'onset_before_h': before['onset_h'],
        'onset_after_h': after['onset_h'],
        'hours_repaired': int(result['replaced'].sum()),
    })
sampling_repair_comparison_004_007_008 = pd.DataFrame(comparison_rows).set_index('LAB')
display(sampling_repair_comparison_004_007_008.round(4))
print('No CO2 calibration/refit was executed; all repairs remain notebook-local experiments.')

### 1b. Activación auditada de las señales sampling-repaired

Se sustituyen únicamente las columnas observadas derivadas de LAB004, LAB007 y LAB008. Las predicciones, timestamps, química, temperatura y restantes LAB permanecen sin cambios. La tabla de procedencia y las aserciones siguientes verifican esta condición antes de continuar.


In [ ]:
REPAIRED_LABS = ['LAB004', 'LAB007', 'LAB008']
nat_original_audit = nat.copy(deep=True)
obs_original_audit = obs.copy(deep=True)
prn_original_audit = prn.copy(deep=True)
prn_old_model_original = prn_old_model.copy(deep=True)

repair_payload = {
    'LAB004': dict(time_h=t004, repaired=q004_repaired, smoothed=q004_repaired_smoothed,
                   windows=repair_windows_lab004, metrics_before=metrics_before, metrics_after=metrics_after),
    'LAB007': dict(time_h=lab007_repair['time_h'], repaired=lab007_repair['repaired'],
                   smoothed=lab007_repair['repaired_smoothed'], windows=lab007_repair['windows'],
                   metrics_before=lab007_repair['metrics_before'], metrics_after=lab007_repair['metrics_after']),
    'LAB008': dict(time_h=lab008_repair['time_h'], repaired=lab008_repair['repaired'],
                   smoothed=lab008_repair['repaired_smoothed'], windows=lab008_repair['windows'],
                   metrics_before=lab008_repair['metrics_before'], metrics_after=lab008_repair['metrics_after']),
}

nat['sampling_repair_applied'] = False
nat['observation_signal_version'] = 'original_current_smoothed'
obs['sampling_repair_applied'] = False
obs['observation_signal_version'] = 'original_current_smoothed'

signal_columns = [
    'co2_rate_filtered_g_l_h', 'co2_rate_robust_median_g_l_h',
    'co2_rate_smoothed_g_l_h', 'co2_rate_g_l_h',
]
for lab in REPAIRED_LABS:
    payload = repair_payload[lab]
    nat_idx = nat.index[nat['batch'].eq(lab)]
    nat_idx = nat.loc[nat_idx].sort_values('t_h').index
    lab_time = nat.loc[nat_idx, 't_h'].to_numpy(dtype=float)
    assert np.allclose(lab_time, payload['time_h'], rtol=0.0, atol=1e-12)

    pulse = pd.DataFrame({'batch': [lab], 'pulse_time_h': [float(bn.loc[lab, 'pulse_time_h'])]})
    smooth_input = nat.loc[nat_idx, ['matrix', 'batch', 't_h']].copy()
    smooth_input['co2_rate_filtered_g_l_h'] = payload['repaired']
    smooth_output = co2x.smooth_hourly_co2_profiles(smooth_input, pulse)
    robust_median = smooth_output['co2_rate_robust_median_g_l_h'].to_numpy(dtype=float)
    assert np.allclose(
        smooth_output['co2_rate_smoothed_g_l_h'].to_numpy(dtype=float),
        payload['smoothed'], rtol=0.0, atol=1e-12,
    )
    replacement_columns = {
        'co2_rate_filtered_g_l_h': payload['repaired'],
        'co2_rate_robust_median_g_l_h': robust_median,
        'co2_rate_smoothed_g_l_h': payload['smoothed'],
        'co2_rate_g_l_h': payload['smoothed'],
    }
    for column, values in replacement_columns.items():
        nat.loc[nat_idx, column] = np.asarray(values, dtype=float)
    nat.loc[nat_idx, 'left_censored'] = (
        nat.loc[nat_idx, 'co2_rate_g_l_h'].to_numpy(dtype=float)
        <= nat.loc[nat_idx, 'detection_limit_g_l_h'].to_numpy(dtype=float)
    )
    nat.loc[nat_idx, 'cold_low_sensitivity'] = (
        nat.loc[nat_idx, 'cold_operation'].astype(bool)
        & nat.loc[nat_idx, 'left_censored'].astype(bool)
    )
    nat.loc[nat_idx, 'sampling_repair_applied'] = True
    nat.loc[nat_idx, 'observation_signal_version'] = 'experimental_repaired_plus_current_smoothing'

    obs_idx = obs.index[(obs['matrix'].eq('natural')) & (obs['batch'].eq(lab))]
    obs_idx = obs.loc[obs_idx].sort_values('t_h').index
    assert np.allclose(obs.loc[obs_idx, 't_h'].to_numpy(dtype=float), lab_time, rtol=0.0, atol=1e-12)
    for column in signal_columns + ['left_censored', 'cold_low_sensitivity']:
        obs.loc[obs_idx, column] = nat.loc[nat_idx, column].to_numpy()
    obs.loc[obs_idx, 'sampling_repair_applied'] = True
    obs.loc[obs_idx, 'observation_signal_version'] = 'experimental_repaired_plus_current_smoothing'

    for prediction_frame in (prn, prn_old_model):
        prn_idx = prediction_frame.index[prediction_frame['batch'].eq(lab)]
        prn_idx = prediction_frame.loc[prn_idx].sort_values('time_h').index
        assert np.allclose(prediction_frame.loc[prn_idx, 'time_h'].to_numpy(dtype=float), lab_time, rtol=0.0, atol=1e-12)
        prediction_frame.loc[prn_idx, 'observed_g_l_h'] = payload['smoothed']
        prediction_frame.loc[prn_idx, 'left_censored'] = nat.loc[nat_idx, 'left_censored'].to_numpy()
    bn.loc[lab, 'observed_onset_h'] = payload['metrics_after']['onset_h']

# Invariantes de alcance: predicciones congeladas y solo tres señales observadas distintas.
assert np.array_equal(
    prn['predicted_g_l_h'].to_numpy(), prn_original_audit['predicted_g_l_h'].to_numpy(), equal_nan=True
)
assert np.array_equal(
    prn_old_model['predicted_g_l_h'].to_numpy(), prn_old_model_original['predicted_g_l_h'].to_numpy(), equal_nan=True
)
changed_labs = []
source_rows = []
for lab in LABS:
    old = nat_original_audit[nat_original_audit['batch'].eq(lab)].sort_values('t_h')
    new = nat[nat['batch'].eq(lab)].sort_values('t_h')
    assert np.allclose(old['t_h'], new['t_h'], rtol=0.0, atol=1e-12)
    max_abs_change = float(np.max(np.abs(
        old['co2_rate_g_l_h'].to_numpy(dtype=float) - new['co2_rate_g_l_h'].to_numpy(dtype=float)
    )))
    changed = max_abs_change > 1e-12
    if changed:
        changed_labs.append(lab)
    if lab not in REPAIRED_LABS:
        for column in signal_columns + ['left_censored', 'cold_low_sensitivity']:
            assert np.array_equal(old[column].to_numpy(), new[column].to_numpy(), equal_nan=True)
    source_rows.append({
        'LAB': lab, 'signal_version': new['observation_signal_version'].iloc[0],
        'sampling_repair_applied': bool(new['sampling_repair_applied'].iloc[0]),
        'max_abs_change_g_l_h': max_abs_change,
    })
assert changed_labs == REPAIRED_LABS
for lab in LABS:
    old_pr = prn_original_audit[prn_original_audit['batch'].eq(lab)].sort_values('time_h')
    new_pr = prn[prn['batch'].eq(lab)].sort_values('time_h')
    if lab not in REPAIRED_LABS:
        assert np.array_equal(old_pr['observed_g_l_h'].to_numpy(), new_pr['observed_g_l_h'].to_numpy(), equal_nan=True)

observation_source_audit = pd.DataFrame(source_rows).set_index('LAB')
observation_source_audit.to_csv(os.path.join(OUT, 'observation_source_audit.csv'))
repair_window_frames = []
for lab in REPAIRED_LABS:
    frame = repair_payload[lab]['windows'].copy()
    frame.insert(0, 'LAB', lab)
    repair_window_frames.append(frame)
sampling_repair_windows_all = pd.concat(repair_window_frames, ignore_index=True)
sampling_repair_windows_all.to_csv(os.path.join(OUT, 'sampling_repair_windows.csv'), index=False)
nat.to_csv(os.path.join(OUT, 'co2_observations_hourly_sampling_repaired.csv'), index=False)
display(observation_source_audit)
print('Changed observed signals:', changed_labs)
print('All other LAB observations are bit-identical; neither frozen prediction set was altered by the repair.')
print('Active prediction set: full-theta natural + recalibrated CO2 + SCCM_CORRECTED.')


In [ ]:
cov = nat.groupby('batch').agg(
    t_ini=('t_h', 'min'), t_fin=('t_h', 'max'), n_horas=('t_h', 'size'),
    cens_frac=('left_censored', 'mean'), art_frac=('artifact_fraction', 'mean'),
    sensor=('sensor_id', 'first'), canal=('acquisition_channel', 'first'),
    quim_ini_h=('chemistry_first_h', 'first'), quim_fin_h=('chemistry_last_h', 'first'))
print('Cobertura canónica (MEDIDO, artefacto SCCM corregido):')
display(cov.round(3))
exc = [ln for ln in open(os.path.join(FM, 'laboratory_2026', 'results', 'co2_matrix_cross_validation_2026', 'excluded_experiments.csv')).read().splitlines() if ln.startswith('natural,LAB009')]
print('LAB009: excluido de la capa CO2 por QC ->', exc[0] if exc else 'NO DISPONIBLE')

In [ ]:
# Metadata estática documentada (fuentes: Maestro_ensayos, Eventos_operacion, Perfiles_temperatura,
# ICS, nombres de archivos de logger, artefacto SCCM). Etiquetas de procedencia por campo.
META = {
 'LAB004': dict(programa='A: 18->21->16', cambios_t='99/171 h', canal='F1', reactor='R-2L-01', sensor_id='1', fecha='2026-04-06', siembra='13:00 (conflicto 10:00/13:00/15:30; logger acota <=14:46)'),
 'LAB005': dict(programa='A: 18->21->16', cambios_t='99/171 h', canal='F2', reactor='R-2L-02', sensor_id='2', fecha='2026-04-06', siembra='13:00 (mismo conflicto que LAB004)'),
 'LAB006': dict(programa='C: 16->18->21', cambios_t='123/219 h', canal='F3', reactor='R-2L-03', sensor_id='3', fecha='2026-04-06', siembra='13:00 (mismo conflicto que LAB004)'),
 'LAB007': dict(programa='B: 21->18->16', cambios_t='72/168 h', canal='F1', reactor='NO DISPONIBLE (sin registro Maestro)', sensor_id='1', fecha='2026-04-17', siembra='16:00 (ICS; Eventos desalineado a 03-25)'),
 'LAB008': dict(programa='A: 18->21->16', cambios_t='96/168 h', canal='F2', reactor='NO DISPONIBLE (sin registro Maestro)', sensor_id='2', fecha='2026-04-17', siembra='16:00 (ICS)'),
 'LAB010': dict(programa='D: 15 isotermo', cambios_t='-', canal='F1', reactor='R-2L-01', sensor_id='1', fecha='2026-05-12', siembra='~15:00 (workbook 10:00 vs ICS 14:45; hojas LAB* anclan 15:00)'),
 'LAB011': dict(programa='E: 18->15->21', cambios_t='24/72 h', canal='F2', reactor='R-2L-02', sensor_id='2', fecha='2026-05-12', siembra='~15:15 (ICS)'),
 'LAB012': dict(programa='F: 15->18->21', cambios_t='60/96 h', canal='F3', reactor='R-2L-03', sensor_id='3', fecha='2026-05-12', siembra='~15:45 (ICS)'),
}
meta = pd.DataFrame(META).T
meta['pulso2_h [INFERIDO cruce rho1040 + fallback ICS]'] = bn['pulse_time_h'].round(1)
meta['rol_capa_CO2'] = bn['role']
print('Asignacion LAB -> canal/fermentador -> sensor_id (MEDIDO desde artefacto y archivos):')
display(meta)
print('NOTA CRITICA: sensor_id y acquisition_channel son 1:1 con el canal F1/F2/F3 en toda la campaña.')
print('No existen calibraciones cruzadas ni permutas de sensor documentadas -> fermentador fisico, canal y sensor SON NO SEPARABLES con esta metadata [NO IDENTIFICABLE].')

In [ ]:
# Quimica inicial y final por LAB desde las hojas LAB004..LAB012 (+LAB009 contexto) del workbook homologado
rows = {}
for lab in LABS + [LAB_CTX]:
    n = int(lab[3:])
    sh = pd.read_excel(XTHIOL, sheet_name='LAB%03d' % n)
    sh['t'] = pd.to_numeric(sh['t'], errors='coerce')
    gnum = pd.to_numeric(sh['GLUCOSE'], errors='coerce')
    fnum = pd.to_numeric(sh['FRUCTOSE'], errors='coerce')
    enum_ = pd.to_numeric(sh['ETANOL'], errors='coerce')
    glynum = pd.to_numeric(sh['GLYCEROL'], errors='coerce')
    r0 = sh[sh.t == 0].iloc[0]
    ig = gnum.notna() & (sh.t > 0)
    last_gf = sh[ig].iloc[-1]
    last_e = sh[enum_.notna() & (sh.t > 0)].iloc[-1]
    last_gly = sh[glynum.notna() & (sh.t > 0)].iloc[-1]
    rows[lab] = dict(GF0=float(r0['GLUCOSE']) + float(r0['FRUCTOSE']),
                     E0_pct=float(r0['ETANOL']), Gly0=float(r0['GLYCEROL']),
                     GF_final=float(last_gf['GLUCOSE']) + float(last_gf['FRUCTOSE']), t_GFfinal=float(last_gf['t']),
                     E_final_pct=float(last_e['ETANOL']), t_Efinal=float(last_e['t']),
                     Gly_final=float(last_gly['GLYCEROL']), t_Glyfinal=float(last_gly['t']))
chem = pd.DataFrame(rows).T
for c in chem.columns:
    chem[c] = pd.to_numeric(chem[c])
bs = pd.read_csv(BATCH_SUMMARY).set_index('batch')
chem['X0'] = bs['X0']
chem['N0'] = bs['N0']
chem['dGF'] = chem.GF0 - chem.GF_final
chem['dE_g'] = (chem.E_final_pct - chem.E0_pct) * ETH_G_PER_PCT
chem['dGly'] = chem.Gly_final - chem.Gly0
print('Quimica [MEDIDO workbook homologado]:')
display(chem.round(3))
print('dGF = GF0 - GF_final; dE_g = (E_final - E0) * 7.8924. Todos los batches (incluido LAB010) terminan')
print('quimicamente secos (GF_final <= 3.3 g/L) con dE ~ 70-73 g/L: la fermentacion fue completa en los 9 casos.')

## 2. Métricas principales por batch

Definiciones (sobre la señal canónica horaria `co2_rate_g_l_h`, SCCM corregido):
- `qmax` / `t_peak`: máximo y tiempo del máximo.
- `integral`: trapezoidal sobre la serie horaria completa (NaN→0).
- `onset`: umbral sostenido objetivo del artefacto canónico (`observed_onset_h`).
- `t_end_act`: último instante con q > 10% del peak; `dur_act = t_end_act - onset`.
- `width_eff = integral / qmax`: ancho efectivo (duración equivalente de amplitud constante qmax).
- T estadística sobre la ventana activa (`sensor_temperature_c`, MEDIDO).
- Índices aparentes (NO estequiometría exacta): `R_sugar = integral/dGF`, `R_eth = integral/dE_g`, `R_rel = R_sugar / mediana(R_sugar)`.
- `v_app_sugar = dGF / dur_act` (velocidad aparente media de consumo).

In [ ]:
met = {}
for lab in LABS:
    g = nat[nat.batch == lab].sort_values('t_h')
    t = g.t_h.values
    q = np.nan_to_num(g.co2_rate_g_l_h.values, nan=0.0)
    pk = q.max()
    tpk = t[q.argmax()]
    integ = float(np.trapz(q, t))
    onset = float(bn.loc[lab, 'observed_onset_h'])
    act = t[q > 0.1 * pk]
    t_end = act.max() if len(act) else t[-1]
    m = (t >= onset) & (t <= t_end)
    Tm = g.sensor_temperature_c.values[m]
    met[lab] = dict(qmax=pk, t_peak=tpk, integral=integ, width_eff=integ / pk, onset=onset,
                    t_end_act=t_end, dur_act=t_end - onset,
                    T_mean=np.nanmean(Tm), T_med=np.nanmedian(Tm), T_min=np.nanmin(Tm), T_max=np.nanmax(Tm),
                    cens_frac=float(g.left_censored.mean()), pulso_h=float(bn.loc[lab, 'pulse_time_h']),
                    canal=str(g.acquisition_channel.iloc[0]), programa=meta.loc[lab, 'programa'])
M = pd.DataFrame(met).T
for c in M.columns:
    if c not in ('canal', 'programa'):
        M[c] = pd.to_numeric(M[c])
M['GF0'] = chem.loc[LABS, 'GF0']
M['dGF'] = chem.loc[LABS, 'dGF']
M['dE_g'] = chem.loc[LABS, 'dE_g']
M['X0'] = chem.loc[LABS, 'X0']
M['N0'] = chem.loc[LABS, 'N0']
M['R_sugar'] = M.integral / M.dGF
M['R_eth'] = M.integral / M.dE_g
M['R_rel'] = M.R_sugar / M.R_sugar.median()
M['qmax_over_X0'] = M.qmax / M.X0
M['v_app_sugar'] = M.dGF / M.dur_act
M.to_csv(os.path.join(OUT, 'master_table_by_batch.csv'))
cols = ['programa', 'canal', 'qmax', 't_peak', 'integral', 'onset', 'dur_act', 'width_eff',
        'T_mean', 'T_med', 'T_min', 'T_max', 'cens_frac', 'X0', 'N0', 'GF0', 'dGF', 'dE_g', 'R_sugar', 'R_eth', 'R_rel']
display(M[cols].round(3))
print('R_sugar ideal (referencia fisica):', round(R_SUGAR_IDEAL, 4), '| R_eth ideal:', round(R_ETH_IDEAL, 4))
print('mediana R_sugar campaña:', round(M.R_sugar.median(), 4), '(magnitud experimental; ideal 0.4886).')
print('La diferencia absoluta vs el ideal NO se interpreta como recuperación física:')
print('incluye convención de conversión/volumen, error analítico y partición de carbono.')
print('matrix_gain es un parámetro empírico de la función de observación del modelo;')
print('NO representa eficiencia de recuperación ni corrige microfugas.')
print('El balance absoluto de carbono NO puede cerrarse con estos datos; R_sugar se usa SOLO relativo.')

## 3. Series temporales (figuras 1–3)

Figura 1: curvas canónicas comparables. Figura 2: curvas normalizadas por su máximo (forma). Figura 3: temperatura medida.

In [ ]:
colors = dict(zip(LABS, plt.cm.tab10(np.linspace(0, 0.75, len(LABS)))))
fig, ax = plt.subplots(figsize=(9.5, 5))
for lab in LABS:
    g = nat[nat.batch == lab].sort_values('t_h')
    ax.plot(g.t_h, g.co2_rate_g_l_h, lw=1.1, color=colors[lab], label='%s (%s, %s)' % (lab, meta.loc[lab, 'programa'], meta.loc[lab, 'canal']))
for lab in LABS:
    ax.plot(M.loc[lab, 't_peak'], M.loc[lab, 'qmax'], 'o', ms=5, color=colors[lab])
ax.set_xlabel('t [h]')
ax.set_ylabel('qCO2 [g/L/h] (SCCM corregido)')
ax.set_title('Fig 1 - Curvas CO2 QC canónicas LAB004-012 (LAB009 excluido por QC)')
ax.legend(ncol=2, fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 5))
for lab in LABS:
    g = nat[nat.batch == lab].sort_values('t_h')
    q = np.nan_to_num(g.co2_rate_g_l_h.values, nan=0.0)
    ax.plot(g.t_h, q / q.max(), lw=1.1, color=colors[lab], label=lab)
ax.set_xlabel('t [h]')
ax.set_ylabel('qCO2 / qmax')
ax.set_title('Fig 2 - Curvas normalizadas por su máximo (comparación de forma)')
ax.legend(ncol=2, fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()
print('Lectura: LAB010 y LAB006 (arranques fríos 15/16 C) son curvas bajas y desplazadas a la derecha;')
print('LAB011/LAB012 (final cálido + X0 alto) dominan en amplitud. Las formas normalizadas son comparables,')
print('sin aplanamientos grotescos ni escalones que sugieran truncamiento del registro.')

In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 4.6))
for lab in LABS:
    g = nat[nat.batch == lab].sort_values('t_h')
    ax.plot(g.t_h, g.sensor_temperature_c, lw=1.0, color=colors[lab], label=lab)
    ax.plot(g.t_h, g.setpoint_c, lw=0.6, ls='--', color=colors[lab], alpha=0.5)
ax.set_xlabel('t [h]')
ax.set_ylabel('T [C] (sólida: medida; punteada: setpoint)')
ax.set_title('Fig 3 - Temperatura medida por batch (MEDIDO, artefacto canónico)')
ax.legend(ncol=2, fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Efecto de temperatura (figuras 4–7)

Correlaciones descriptivas y regresiones simples con `T_mean` (temperatura media medida en ventana activa) y con el setpoint dominante. n=8 y un solo programa térmico claramente replicado: lectura descriptiva; una correlación NO demuestra causalidad. La asociación T↔qmax/velocidad es fuerte y físicamente esperable; en cambio, que el **CO2 total recuperado** correlacione con T NO tiene explicación fisiológica directa (a química igual el total debería ser ~constante) y se examina críticamente en §6/§18. `T_mean` en ventana activa es el resumen justificado porque el peak ocurre dentro de esa ventana.

In [ ]:
def pearson(x, y):
    return float(np.corrcoef(np.asarray(x, float), np.asarray(y, float))[0, 1])

pairs = [('qmax', 'T_mean'), ('t_peak', 'T_mean'), ('dur_act', 'T_mean'), ('integral', 'T_mean')]
fig, axes = plt.subplots(2, 2, figsize=(10, 7.5))
for ax, (yv, xv) in zip(axes.ravel(), pairs):
    x = M[xv].values
    y = M[yv].values
    for lab in LABS:
        ax.plot(M.loc[lab, xv], M.loc[lab, yv], 'o', ms=9, color=colors[lab])
        ax.annotate(lab, (M.loc[lab, xv], M.loc[lab, yv]), textcoords='offset points', xytext=(6, 4), fontsize=8)
    b1 = np.polyfit(x, y, 1)
    xs = np.linspace(x.min(), x.max(), 10)
    ax.plot(xs, np.polyval(b1, xs), 'k--', lw=1)
    r = pearson(x, y)
    r_no10 = pearson(np.delete(x, LABS.index('LAB010')), np.delete(y, LABS.index('LAB010')))
    ax.set_xlabel('%s (T media ventana activa)' % xv)
    ax.set_ylabel(yv)
    ax.set_title('%s vs T_mean | r=%.2f (sin LAB010: r=%.2f)' % (yv, r, r_no10), fontsize=10)
    ax.grid(alpha=0.3)
fig.suptitle('Fig 4-7 - Relaciones con temperatura medida (n=8, descriptivo)', y=1.0)
fig.tight_layout()
plt.show()
for yv, xv in pairs:
    print('%10s vs T_mean: r=%+.3f' % (yv, pearson(M[xv], M[yv])))
print()
print('Lectura: qmax crece fuertemente con T (patrón cinético esperado).')
print('ATENCION: integral tambien correlaciona con T (r=%+.3f). Dado que todos los batches' % pearson(M.T_mean, M.integral))
print('consumieron azúcar y produjeron etanol casi iguales, el TOTAL de gas debería ser ~independiente')
print('de T: esa correlación es físicamente sospechosa y refleja que los batches fríos (LAB006/010)')
print('son exactamente los de menor recuperación relativa -> ver sección 6 (balance vs química).')

## 5. Peak vs área: ¿los batches de peak bajo son solo más anchos/lentos? (figura 8)

Patrón A (peak distinto, área similar → cinética distinta) vs patrón B (peak bajo + área anormalmente baja pese a química comparable → pérdida de gas/instrumentación).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.4))
def scat(ax, xcol, ycol, xlab, ylab, title):
    for lab in LABS:
        ax.plot(M.loc[lab, xcol], M.loc[lab, ycol], 'o', ms=9, color=colors[lab])
        ax.annotate(lab, (M.loc[lab, xcol], M.loc[lab, ycol]), textcoords='offset points', xytext=(6, 4), fontsize=8)
    ax.set_xlabel(xlab)
    ax.set_ylabel(ylab)
    ax.set_title(title, fontsize=10)
    ax.grid(alpha=0.3)
scat(axes[0], 'integral', 'qmax', 'integral CO2 [g/L]', 'qmax [g/L/h]', 'Fig 8a - qmax vs integral')
scat(axes[1], 'width_eff', 'qmax', 'ancho efectivo integral/qmax [h]', 'qmax [g/L/h]', 'Fig 8b - ancho vs qmax')
scat(axes[2], 'dur_act', 'qmax', 'duración activa [h]', 'qmax [g/L/h]', 'Fig 8c - duración vs qmax')
fig.tight_layout()
plt.show()
print('qmax vs integral: r=%+.3f | qmax vs width_eff: r=%+.3f | qmax vs dur_act: r=%+.3f' % (
    pearson(M.integral, M.qmax), pearson(M.width_eff, M.qmax), pearson(M.dur_act, M.qmax)))
print()
print('Lectura A/B:')
print('- LAB006 (arranque 16 C): peak bajo PERO ancho efectivo alto (80 h) -> patrón A (más lenta/ancho). Su déficit de integral vs modelo se explica por la cola predicha (§15): no por su peak.')
print('- LAB010 (15 C): peak bajo SIN ancho compensatorio (width_eff ~57 h, en el rango de los batches')
print('  cálidos) y con la integral más baja de la campaña -> patrón B: NO es simplemente más ancha.')
print('- LAB011/LAB012: picos altos con anchos cortos (40-50 h) -> fermentaciones rápidas completas.')
print('RESULTADO DEL CÁLCULO: el ranking de integral NO replica al de qmax; la dispersión de integral')
print('queda dominada por LAB010 (y en menor grado LAB006), no por un gradiente térmico global.')

## 6. Balance de CO2 gaseoso vs química (figuras 9–11) — prueba central

`R_sugar = integral / dGF` y `R_eth = integral / dE_g` son **índices aparentes de recuperación**, NO balances estequiométricos exactos: parte del carbono va a biomasa, glicerol (dGly medido 4.4–5.9 g/L), metabolitos secundarios y CO2 disuelto (~1–2 g/L), y la química tiene error experimental. La estequiometría ideal C6H12O6 → 2 CO2 + 2 EtOH (0.4886 g CO2/g azúcar; 0.955 g CO2/g etanol) se usa SOLO como referencia física.

**Magnitudes separadas que NO deben confundirse**: (A) la medición experimental `R_sugar = integral/Δ(G+F)` (mediana ~0.22); (B) la referencia estequiométrica ideal (0.4886); (C) `matrix_gain`, un parámetro **empírico de la función de observación del modelo** que escala qgas→predicción y cuyo valor se toma de la recalibración full-theta. `matrix_gain` NO representa eficiencia de recuperación física, no mide porcentaje perdido y no corrige microfugas. La diferencia absoluta entre (A) y (B) no puede atribuirse a una causa única (convención/volumen, error analítico, carbono a biomasa/glicerol/metabolitos/CO2 disuelto) y **el balance absoluto de carbono no puede cerrarse con estos datos**. Lo decisivo es la consistencia RELATIVA entre batches: dos batches con química comparable (dGF y dE casi idénticos, fermentación completa) pero con gas integrado muy distinto constituyen evidencia compatible con pérdida de gas o lectura instrumental baja, sin demostrarla.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13.8, 4.6))
ax = axes[0]
for lab in LABS:
    ax.plot(M.loc[lab, 'dGF'], M.loc[lab, 'integral'], 'o', ms=9, color=colors[lab])
    ax.annotate(lab, (M.loc[lab, 'dGF'], M.loc[lab, 'integral']), textcoords='offset points', xytext=(6, 4), fontsize=8)
xx = np.linspace(M.dGF.min() * 0.98, M.dGF.max() * 1.01, 10)
ax.plot(xx, R_SUGAR_IDEAL * xx, 'k--', lw=1, label='ideal 0.489')
ax.plot(xx, M.R_sugar.median() * xx, 'g--', lw=1.2, label='mediana campaña %.3f' % M.R_sugar.median())
ax.set_xlabel('azúcar consumida dGF [g/L]')
ax.set_ylabel('integral CO2 gas [g/L]')
ax.set_title('Fig 9 - integral vs d(G+F)', fontsize=10)
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
ax = axes[1]
for lab in LABS:
    ax.plot(M.loc[lab, 'dE_g'], M.loc[lab, 'integral'], 'o', ms=9, color=colors[lab])
    ax.annotate(lab, (M.loc[lab, 'dE_g'], M.loc[lab, 'integral']), textcoords='offset points', xytext=(6, 4), fontsize=8)
xx = np.linspace(M.dE_g.min() * 0.98, M.dE_g.max() * 1.01, 10)
ax.plot(xx, R_ETH_IDEAL * xx, 'k--', lw=1, label='ideal 0.955')
ax.plot(xx, M.R_eth.median() * xx, 'g--', lw=1.2, label='mediana campaña %.3f' % M.R_eth.median())
ax.set_xlabel('etanol producido dE [g/L]')
ax.set_ylabel('integral CO2 gas [g/L]')
ax.set_title('Fig 11 - integral vs dE (E final MEDIDO en los 8)', fontsize=10)
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
ax = axes[2]
order = M.sort_values('R_rel').index.tolist()
ax.bar(range(len(order)), M.loc[order, 'R_rel'], color=[colors[l] for l in order])
ax.axhline(1.0, color='k', lw=1)
ax.axhspan(0.8, 1.2, color='grey', alpha=0.15)
ax.set_xticks(range(len(order)))
ax.set_xticklabels(order, rotation=60, fontsize=8)
ax.set_ylabel('R_sugar / mediana(R_sugar)')
ax.set_title('Fig 10/15 - recuperación relativa por LAB (banda 0.8-1.2)', fontsize=10)
ax.grid(alpha=0.3, axis='y')
fig.tight_layout()
plt.show()
disp = chem.loc[LABS, ['GF_final', 'dGF', 'dE_g']].join(M[['integral', 'R_sugar', 'R_eth', 'R_rel']])
disp['dGly'] = chem.loc[LABS, 'dGly']
display(disp.round(3))
print('dGF varía %.1f-%.1f (CV %.1f%%) y dE %.1f-%.1f (CV %.1f%%): química final casi idéntica en los 8.' % (
    M.dGF.min(), M.dGF.max(), 100 * M.dGF.std() / M.dGF.mean(), M.dE_g.min(), M.dE_g.max(), 100 * M.dE_g.std() / M.dE_g.mean()))
print('integral varía %.1f-%.1f (CV %.1f%%): la dispersión del gas NO está explicada por la química.' % (
    M.integral.min(), M.integral.max(), 100 * M.integral.std() / M.integral.mean()))
print()
print('RESULTADO DEL CÁLCULO: R_rel -> ' + ', '.join('%s %.2f' % (l, M.loc[l, 'R_rel']) for l in LABS))
others = M.drop(index=['LAB010', 'LAB006'])['R_rel']
print('LAB010 = %.2f y LAB006 = %.2f quedan fuera de la banda 0.8-1.2; el resto %.2f-%.2f.' % (M.loc['LAB010', 'R_rel'], M.loc['LAB006', 'R_rel'], others.min(), others.max()))

### 6b. Lectura del balance sampling-repaired

- [HECHO MEDIDO] Los 8 batches mantienen la misma química: consumieron aproximadamente 146-166 g/L de azúcar y produjeron 70-73 g/L de etanol.
- [RESULTADO DEL CÁLCULO] El gas integrado sigue abarcando 14.7-41.8 g/L; reparar tres LAB no elimina la dispersión global.
- [RESULTADO DEL CÁLCULO] LAB004 y LAB007 aumentan su integral; LAB008 queda prácticamente invariante porque se eliminan tanto caídas como rebotes.
- [RESULTADO DEL CÁLCULO] LAB010 y LAB006 siguen siendo las recuperaciones relativas más bajas. Sus señales no fueron modificadas.
- [INTERPRETACIÓN] La evidencia de baja recuperación asociada a depresiones de muestreo se debilita para LAB004 y no aparece para LAB007/LAB008; esto no demuestra ausencia de fuga.


## 7. Normalizaciones

Robustas (denominadores medidos y comparables): `integral/dGF`, `integral/dE`, `qmax/dGF`. Exploratorias (denominador con semántica dudosa o ruido): `qmax/X0` (X0 depende del protocolo de inoculación y de la semántica Oculyze; LAB011/012 tienen X0 2–10× mayor).

In [ ]:
norm = pd.DataFrame({
    'qmax_over_dGF': M.qmax / M.dGF,
    'integral_over_dGF': M.R_sugar,
    'integral_over_dE': M.R_eth,
    'qmax_over_X0 [EXPLORATORIO]': M.qmax_over_X0,
    'v_app_sugar': M.v_app_sugar,
    'qmax_over_Tmean': M.qmax / M.T_mean})
display(norm.round(4))
print('Correlaciones con T_mean: qmax/dGF r=%+.3f | v_app r=%+.3f | qmax/X0 r=%+.3f' % (
    pearson(M.T_mean, norm['qmax_over_dGF']), pearson(M.T_mean, norm['v_app_sugar']), pearson(M.T_mean, M['qmax_over_X0'])))
print('Lectura: la velocidad aparente (v_app, qmax/dGF) sigue la temperatura como era de esperar;')
print('qmax/X0 no ordena los datos (X0 alto de LAB011/012 coexiste con picos altos pero LAB004/007 con')
print('X0 0.29-0.39 también tienen picos medios-altos): X0 no es un normalizador confiable aquí [EXPLORATORIO].')

## 8. Fermentador / sensor / canal (figura 13)

[HECHO MEDIDO] La asignación es F1→LAB004/007/010, F2→LAB005/008/011, F3→LAB006/009(ctx)/012, con `sensor_id` 1/2/3 fijo por canal y sin permutas documentadas. [NO IDENTIFICABLE] Con esta metadata, fermentador físico, canal y sensor NO son separables: cada uno sigue exactamente al mismo grupo. La prueba disponible es la consistencia de métricas de un mismo canal a través de fechas distintas.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.3))
for ax, col, ylab in zip(axes, ['qmax', 'integral', 'R_rel'], ['qmax [g/L/h]', 'integral [g/L]', 'R_sugar relativo']):
    for canal in ['F1', 'F2', 'F3']:
        sub = M[M.canal == canal].sort_values('fecha' if 'fecha' in M else 't_peak')
        sub = M[M.canal == canal]
        x = [meta.loc[l, 'fecha'] for l in sub.index]
        ax.plot(x, sub[col], 'o-', ms=8, label=canal)
        for l in sub.index:
            ax.annotate(l, (meta.loc[l, 'fecha'], sub.loc[l, col]), textcoords='offset points', xytext=(5, 4), fontsize=8)
    ax.set_ylabel(ylab)
    ax.set_title('por canal a través de las fechas', fontsize=10)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
fig.suptitle('Fig 13 - Métricas por canal/fermentador (F1, F2, F3) vs fecha', y=1.02)
fig.tight_layout()
plt.show()
print(M.groupby('canal')[['qmax', 'integral', 'R_rel']].agg(['min', 'max', 'mean']).round(3))
print()
print('Lectura: cada canal alcanza valores altos y bajos en distintas fechas -> NO hay offset constante')
print('por canal que sugiera calibración sesgada de un sensor específico en toda la campaña.')
print('La única anomalía fuerte (LAB010, R_rel ~0.45) ocurre en F1 en mayo; F1 en abril (LAB004/007, R_rel ~1)')
print('fue normal -> la anomalía es específica de esa fecha/canal/reactor, no del canal en general.')

In [ ]:
# Par replicado ideal: LAB004 vs LAB005 (mismo día, mismo mosto, mismo programa A; solo cambia F1/F2)
fig, ax = plt.subplots(figsize=(9.5, 4.4))
for lab in ['LAB004', 'LAB005']:
    g = nat[nat.batch == lab].sort_values('t_h')
    ax.plot(g.t_h, g.co2_rate_g_l_h, lw=1.3, color=colors[lab], label=lab)
ax.set_xlabel('t [h]')
ax.set_ylabel('qCO2 [g/L/h]')
ax.set_title('Réplica nominal LAB004 (F1) vs LAB005 (F2): mismo día, mismo programa A')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()
d = (M.loc['LAB004', ['qmax', 'integral', 'R_rel']] - M.loc['LAB005', ['qmax', 'integral', 'R_rel']]) / M.loc['LAB005', ['qmax', 'integral', 'R_rel']]
print('Diferencias relativas LAB004 vs LAB005: ' + ', '.join('%s %+.1f%%' % (k, 100 * v) for k, v in d.items()))
print('HECHO NUMERICO: tras reparar LAB004, sus diferencias vs LAB005 son las mostradas arriba.')
print('INTERPRETACION: ambas integrales siguen en el grupo normal, pero integral y R_rel ya no difieren <4%;')
print('la concordancia nominal LAB004/LAB005 se debilita y no debe describirse como replica casi identica.')


## 9. Comparaciones entre réplicas térmicas (figura 12)

El único programa replicado es A (18→21→16): LAB004 y LAB005 (mismo día) y LAB008 (11 días después). B, C, D, E y F son programas únicos. Estas comparaciones intra-programa pesan más que las correlaciones globales.

In [ ]:
grupoA = ['LAB004', 'LAB005', 'LAB008']
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.4))
for lab in grupoA:
    g = nat[nat.batch == lab].sort_values('t_h')
    axes[0].plot(g.t_h, g.co2_rate_g_l_h, lw=1.2, color=colors[lab], label='%s (%s)' % (lab, meta.loc[lab, 'fecha']))
axes[0].set_xlabel('t [h]')
axes[0].set_ylabel('qCO2 [g/L/h]')
axes[0].set_title('Fig 12a - Grupo térmico A (18->21->16)')
axes[0].legend(fontsize=9)
axes[0].grid(alpha=0.3)
w = 0.25
for i, (lab, col) in enumerate([('qmax', 'qmax [g/L/h]'), ('integral', 'integral [g/L]'), ('R_rel', 'R_rel')]):
    axes[1].bar(np.arange(3) + (i - 1) * w, M.loc[grupoA, lab].values, width=w, label=col)
axes[1].set_xticks(range(3))
axes[1].set_xticklabels(grupoA)
axes[1].legend(fontsize=8)
axes[1].set_title('Fig 12b - Métricas del grupo A')
axes[1].grid(alpha=0.3, axis='y')
fig.tight_layout()
plt.show()
print(M.loc[grupoA, ['qmax', 't_peak', 'integral', 'R_rel', 'X0', 'T_mean']].round(3))
spread_q = 100.0 * (M.loc[grupoA, 'qmax'].max() / M.loc[grupoA, 'qmax'].min() - 1.0)
spread_i = 100.0 * (M.loc[grupoA, 'integral'].max() / M.loc[grupoA, 'integral'].min() - 1.0)
print('Grupo A reparado: qmax %.3f-%.3f (spread %.1f%%), integral %.1f-%.1f (spread %.1f%%), R_rel %.2f-%.2f.' % (
    M.loc[grupoA, 'qmax'].min(), M.loc[grupoA, 'qmax'].max(), spread_q,
    M.loc[grupoA, 'integral'].min(), M.loc[grupoA, 'integral'].max(), spread_i,
    M.loc[grupoA, 'R_rel'].min(), M.loc[grupoA, 'R_rel'].max()))
print('INTERPRETACION: el grupo A sigue en el rango normal, pero es menos estrecho que en la auditoria original.')


## 10. Discontinuidades en las señales (raw/filtrado, partes del logger)

Se reconstruyen los límites de cada parte de logger por LAB (primera/última marca de tiempo de cada archivo crudo) para detectar huecos de adquisición, y se buscan escalones persistentes en la señal filtrada horaria (mediana móvil 5 h; cambio sostenido >= 0.08 g/L/h). Un escalón o caída no explicada por temperatura/química es candidato a evento de proceso/instrumento — NO se clasifica automáticamente como fuga.

In [ ]:
PARTES = {
 'LAB004': [('CO2_F1_lab004.csv', DESK), ('CO2_F1_LAB004_PT2.csv', RAW)],
 'LAB005': [('CO2_F2_lab005.csv', DESK), ('CO2_F2_LAB005_PT2.csv', RAW)],
 'LAB006': [('CO2_F3_lab006.csv', DESK), ('CO2_F3_LAB006_PT2.csv', RAW)],
 'LAB007': [('CO2_F1_lab007.csv', RAW)],
 'LAB008': [('CO2_F2_lab008.csv', RAW)],
 'LAB009': [('CO2_F3_lab009.csv', RAW)],
 'LAB010': [('CO2_F1_lab010.csv', RAW), ('CO2_F1_lab10_PT2.csv', RAW)],
 'LAB011': [('CO2_F2_lab011.csv', RAW), ('CO2_F2_lab11_PT2..csv', RAW)],
 'LAB012': [('CO2_F3_lab012.csv', RAW), ('CO2_F3_lab12_PT2.csv', RAW)],
}
rows = []
for lab, partes in PARTES.items():
    bounds = []
    for fname, folder in partes:
        path = os.path.join(folder, fname)
        with open(path, encoding='utf-8-sig', errors='replace') as fh:
            lines = [l for l in fh.read().splitlines() if l.strip()]
        bounds.append((fname, lines[1].split(',')[0], lines[-1].split(',')[0], len(lines) - 1))
    for i, (fname, t0f, t1f, n) in enumerate(bounds):
        gap = ''
        if i > 0:
            dt = (pd.Timestamp(t0f) - pd.Timestamp(bounds[i - 1][2])).total_seconds() / 3600.0
            gap = '%.2f h' % dt
        rows.append(dict(LAB=lab, archivo=fname, primera=t0f, ultima=t1f, n_lineas=n, hueco_previo=gap))
parts = pd.DataFrame(rows)
display(parts)
print('Huecos de adquisición entre partes: LAB004-006 ~1.15 h (07:13->08:22 del 07-04); LAB010-012 ~2.7 h (11:59->14:39 del 13-05).')
print('Pérdida máxima de integral atribuible a un hueco: ~2.7 h x q (~0.2-0.9 g/L/h) = 0.5-2.4 g/L: NO explica las diferencias de 15-27 g/L.')

In [ ]:
steps = []
for lab in LABS:
    g = nat[nat.batch == lab].sort_values('t_h')
    t = g.t_h.values
    qs = pd.Series(np.nan_to_num(g.co2_rate_filtered_g_l_h.values, nan=0.0)).rolling(5, center=True, min_periods=3).median()
    onset = float(bn.loc[lab, 'observed_onset_h'])
    d = qs.diff()
    for i in range(2, len(d) - 8):
        if abs(d.iloc[i]) >= 0.08 and t[i] > onset + 6:
            before = float(qs.iloc[max(0, i - 8):i].median())
            after = float(qs.iloc[i + 1:i + 9].median())
            if abs(after - before) >= 0.08:
                steps.append(dict(LAB=lab, t_h=round(float(t[i]), 1), nivel_antes=round(before, 3),
                                  nivel_despues=round(after, 3), salto=round(after - before, 3)))
steps_df = pd.DataFrame(steps)
steps_df.to_csv(os.path.join(OUT, 'discontinuity_events.csv'), index=False)
display(steps_df)
print('Lectura: los saltos detectados se concentran alrededor de los pulsos de nutrición y cambios de')
print('setpoint y ventanas de muestreo (08:00/15:00-16:00 diarias: transiciones reales del proceso).')
print('No se detectan caídas persistentes bruscas INCOMPATIBLES')
print('con temperatura/química en la señal filtrada de ningún batch aceptado.')

## 11. LAB009 como contexto diagnóstico (cadena NO canónica)

LAB009 (F3, 17-04, programa A) fue excluido de la capa CO2 por QC (`discarded: unreliable CO2 implementation/profile`). Aquí se procesa su CO2 crudo con una cadena simple y NO canónica (mediana móvil 61 puntos ~10 min + conversión SCCM corregida + bin horario), solo para comparación visual y métricas de contexto. **No se mezcla con los batches aceptados en ninguna estadística.**

In [ ]:
d9 = pd.read_csv(os.path.join(RAW, 'CO2_F3_lab009.csv'))
d9['ts'] = pd.to_datetime(d9.timestamp)
d9 = d9.sort_values('ts').reset_index(drop=True)
t0_9 = pd.Timestamp('2026-04-17 16:00:00')
t9 = (d9.ts - t0_9).dt.total_seconds().values / 3600.0
q9 = d9.flow_sccm.rolling(61, center=True, min_periods=20).median() * SCCM_CORRECTED
bins = np.arange(np.floor(t9.min()), np.ceil(t9.max()) + 1)
idx = np.clip(np.digitize(t9, bins) - 1, 0, len(bins) - 2)
qh = pd.Series(np.nan_to_num(q9.values)).groupby(idx).mean()
th = 0.5 * (bins[:-1] + bins[1:])
int9 = float(np.trapz(qh.values, th))
pk9 = float(qh.max())
dGF9 = float(chem.loc['LAB009', 'dGF'])
print('LAB009 [NO CANONICO]: peak=%.3f g/L/h, integral=%.1f g/L, R_sugar=%.3f (R_rel=%.2f vs mediana aceptados)' % (
    pk9, int9, int9 / dGF9, (int9 / dGF9) / M.R_sugar.median()))
fig, ax = plt.subplots(figsize=(9.5, 4.4))
for lab in ['LAB006', 'LAB012']:
    g = nat[nat.batch == lab].sort_values('t_h')
    ax.plot(g.t_h, g.co2_rate_g_l_h, lw=1.2, color=colors[lab], label='%s (canónico, F3)' % lab)
ax.plot(th, qh.values, lw=1.2, ls='--', color='k', label='LAB009 (NO canónico, F3)')
ax.set_xlabel('t [h]')
ax.set_ylabel('qCO2 [g/L/h]')
ax.set_title('LAB009 en contexto de sus hermanos de canal F3')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()
print('Lectura: la señal de LAB009 tiene forma plausible PERO recuperación aparente baja (R_rel ~0.44).')
print('Esta cadena NO resta el offset de cero (+0.86 sccm pre-inóculo documentado en abril): la recuperación')
print('equivalente canónica sería aún menor. Consistente con la exclusión por QC (unreliable implementation/profile);')
print('NO comparable directamente con los aceptados. Se mantiene excluido por decisión QC canónica.')

## 12. Modelo congelado como referencia auxiliar (figura 14)

Se usan las predicciones ya versionadas de `co2_matrix_cross_validation_2026_full_theta_sccm_corrected` (theta natural completo → capa CO2 natural recalibrada; sin fitting en este notebook). `ratio_peak = peak_obs/peak_pred`, `ratio_int = integral_obs/integral_pred`. El RMSE conserva el soporte no censurado de la calibración.

ADVERTENCIA — `int_obs / int_pred < 1` NO es evidencia directa de microfuga. Puede provenir de: upstream incorrecto, terminación demasiado lenta del modelo, gate de activación, respuesta nutricional, dinámica de transferencia, error del modelo, sensor, pérdida física de gas, o combinaciones. El modelo congelado es una línea AUXILIAR: no se usa como prueba principal de fuga. Las secciones 14–17 cuantifican cada una de esas contribuciones.

In [ ]:
rat = []
for lab in LABS:
    g = prn[prn.batch == lab].sort_values('time_h')
    o = np.nan_to_num(g.observed_g_l_h.values, nan=0.0)
    p = np.nan_to_num(g.predicted_g_l_h.values, nan=0.0)
    valid = ~g.left_censored.astype(bool).to_numpy()
    t = g.time_h.to_numpy(dtype=float)
    rat.append(dict(LAB=lab, peak_obs=o.max(), peak_pred=p.max(), ratio_peak=o.max() / p.max(),
                    t_peak_obs=float(t[o.argmax()]), t_peak_pred=float(t[p.argmax()]),
                    int_obs=float(np.trapz(o, t)), int_pred=float(np.trapz(p, t)),
                    rmse_g_l_h=float(np.sqrt(np.mean((o[valid] - p[valid]) ** 2))), n_rmse=int(valid.sum())))
R = pd.DataFrame(rat).set_index('LAB')
R['ratio_int'] = R.int_obs / R.int_pred
R.to_csv(os.path.join(OUT, 'observed_vs_predicted_metrics_full_theta.csv'))
display(R.round(3))
fig, axes = plt.subplots(2, 4, figsize=(14, 6.6), sharex=False)
for ax, lab in zip(axes.ravel(), LABS):
    g = prn[prn.batch == lab].sort_values('time_h')
    ax.plot(g.time_h, g.observed_g_l_h, lw=1.1, color=colors[lab], label='obs')
    ax.plot(g.time_h, g.predicted_g_l_h, lw=1.1, ls='--', color='k', label='pred (full-theta congelado)')
    ax.set_title('%s | ratio_peak %.2f | ratio_int %.2f' % (lab, R.loc[lab, 'ratio_peak'], R.loc[lab, 'ratio_int']), fontsize=9)
    ax.set_xlabel('t [h]')
    ax.grid(alpha=0.3)
    ax.legend(fontsize=7)
fig.suptitle('Fig 14 - Observación vs predicción congelada full-theta + CO2 recalibrado', y=1.0)
fig.tight_layout()
plt.show()
for lab in LABS:
    row = R.loc[lab]
    print('%s: ratio_peak %.3f | ratio_int %.3f | RMSE %.3f | t_peak obs/pred %.0f/%.0f h' % (
        lab, row.ratio_peak, row.ratio_int, row.rmse_g_l_h, row.t_peak_obs, row.t_peak_pred))
print('Lectura: la referencia full-theta cambia magnitud y timing predichos, pero sigue siendo una línea auxiliar, no una prueba de fuga.')
print('LAB011/012 mantienen CO2 observado por encima del predicho; la comparación formal con la referencia anterior se presenta al final.')

## 13. Tabla conceptual de firmas de las hipótesis con señales sampling-repaired

| Hipótesis | Firma esperada | Evidencia recalculada |
|---|---|---|
| Temperatura/cinética | qmax asociado con T; frío = peak menor y curva más ancha | PRESENTE PARCIALMENTE: r(qmax,T) ~ +0.87; LAB006 mantiene peak bajo, ancho ~80 h y R_rel ~0.66 |
| Microfuga/pérdida de gas | integral anormalmente baja con química comparable | SOLO LAB010 conserva esta firma entre aceptados (R_rel ~0.45); el modelo no la corrobora y no separa fuga de lectura baja |
| Sensor/canal | anomalías persistentes por canal | NO SOPORTADA como sesgo permanente; cada canal mantiene valores altos y bajos según fecha |
| Biología/ICs | gas y química cambian coherentemente | NO CONFIRMADA para explicar el exceso LAB011/LAB012; los contrafactuales congelados siguen sin reproducirlo |
| Defecto estructural del modelo | gate no causal, cola lenta, respuesta a pulso atenuada | PRESENTE con full-theta: la cola explica 94% del déficit de LAB006; el gate recorta fuertemente LAB010/011/012; no prueba microfugas |
| Artefactos de muestreo | caídas/spikes alrededor de aperturas registradas | REDUCIDOS en LAB004/LAB007/LAB008; las discontinuidades persistentes restantes se recalculan sobre las señales reparadas |

La tabla clasifica compatibilidad de evidencia, no causalidad demostrada.


## 14. Auditoría de las ventanas de integración (`int_obs` / `int_pred`)

Antes de interpretar discrepancias: ¿sobre qué ventanas se integraron observación y predicción? En el
artefacto canónico, `predict_and_score` evalúa `predicted = matrix_gain * qgas` **en los mismos
timestamps horarios de la observación** (`_metric_row` integra ambos con `trapz` sobre ese `time_h`),
por lo que las ventanas son idénticas por construcción (caso A). Aquí se verifica y se auditan NaN.

In [ ]:
audit_rows = []
for lab in LABS:
    g = prn[prn.batch == lab].sort_values('time_h')
    o = g.observed_g_l_h.values
    p = g.predicted_g_l_h.values
    audit_rows.append(dict(LAB=lab,
        t_obs_start=float(g.time_h.min()), t_obs_end=float(g.time_h.max()),
        t_pred_start=float(g.time_h.min()), t_pred_end=float(g.time_h.max()),
        integration_start=float(g.time_h.min()), integration_end=float(g.time_h.max()),
        n_nan_obs=int(np.isnan(o).sum()), n_nan_pred=int(np.isnan(p).sum()),
        int_obs=float(np.trapz(np.nan_to_num(o), g.time_h.values)),
        int_pred=float(np.trapz(np.nan_to_num(p), g.time_h.values))))
AUD = pd.DataFrame(audit_rows).set_index('LAB')
AUD['ratio_int'] = AUD.int_obs / AUD.int_pred
display(AUD.round(3))
print('VEREDICTO (A): observación y predicción se integran sobre exactamente la misma ventana y grilla')
print('horaria (mismos timestamps, sin interpolación ni extrapolación). Sin NaN en ninguna de las dos series.')
print('Nota: la grilla del DRIVER (0.25 h hasta chemistry_last_h) es interna del modelo; la comparación')
print('canónica ocurre en la grilla de observación. No se corrige nada: el cálculo ya es homogéneo.')

## 15. Peak vs cola: ¿la diferencia de integral es amplitud o terminación?

Definiciones declaradas (reproducibles, independientes del resultado):
- **Terminación** `t_end`: primer instante a partir del cual la señal permanece < 5% de su máximo hasta el fin del registro.
- **Cola**: integral post-peak (desde el propio t_peak de cada serie). `tail_excess = post_peak_pred - post_peak_obs`.
Caveat: las ventanas post-peak difieren si t_peak_obs != t_peak_pred; se reporta `dt_end` como contexto.

In [ ]:
def t_end_5pct(t, q, tpk):
    thr = 0.05 * q.max()
    m = t >= tpk
    tt, qq = t[m], q[m]
    below = np.where(qq < thr)[0]
    for i in range(len(below)):
        if np.all(qq[below[i]:] < thr):
            return float(tt[below[i]])
    return float(tt[-1])

trows = []
for lab in LABS:
    g = prn[prn.batch == lab].sort_values('time_h')
    tt = g.time_h.values
    o = np.nan_to_num(g.observed_g_l_h.values)
    p = np.nan_to_num(g.predicted_g_l_h.values)
    pk_o, pk_p = o.max(), p.max()
    tpk_o, tpk_p = float(tt[o.argmax()]), float(tt[p.argmax()])
    trows.append(dict(LAB=lab, peak_obs=pk_o, peak_pred=pk_p, ratio_peak=pk_o / pk_p,
        t_peak_obs=tpk_o, t_peak_pred=tpk_p,
        t_end_obs=t_end_5pct(tt, o, tpk_o), t_end_pred=t_end_5pct(tt, p, tpk_p),
        post_peak_obs=float(np.trapz(o[tt >= tpk_o], tt[tt >= tpk_o])),
        post_peak_pred=float(np.trapz(p[tt >= tpk_p], tt[tt >= tpk_p]))))
TAIL = pd.DataFrame(trows).set_index('LAB')
TAIL['delta_t_end'] = TAIL.t_end_pred - TAIL.t_end_obs
TAIL['tail_excess'] = TAIL.post_peak_pred - TAIL.post_peak_obs
TAIL['deficit_int'] = AUD.int_pred - AUD.int_obs
TAIL['tail_share_of_deficit'] = TAIL.tail_excess / TAIL.deficit_int
TAIL.to_csv(os.path.join(OUT, 'tail_decomposition_by_lab.csv'))
display(TAIL.round(2))
for lab in LABS:
    row = TAIL.loc[lab]
    print('%s: tail_excess %+.2f g/L | déficit integral %+.2f g/L | tail_share %+.2f' % (
        lab, row.tail_excess, row.deficit_int, row.tail_share_of_deficit))
print('Lectura: el signo y la magnitud de cola se reinterpretan con la predicción full-theta; no se convierten automáticamente en evidencia de fuga.')

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(14.5, 6.8), sharex=False)
for ax, lab in zip(axes.ravel(), LABS):
    g = prn[prn.batch == lab].sort_values('time_h')
    tt = g.time_h.values
    o = np.nan_to_num(g.observed_g_l_h.values)
    p = np.nan_to_num(g.predicted_g_l_h.values)
    te = TAIL.loc[lab, 't_end_obs']
    ax.plot(tt, o, lw=1.2, color=colors[lab], label='obs')
    ax.plot(tt, p, lw=1.2, ls='--', color='k', label='pred (congelado)')
    msk = (p > 0) & (tt >= TAIL.loc[lab, 't_end_pred'])
    ax.fill_between(tt[msk], 0, p[msk], color='grey', alpha=0.35, label='cola predicha')
    ax.set_title('%s | ratio_int %.2f | tail_excess %+.1f' % (lab, AUD.loc[lab, 'ratio_int'], TAIL.loc[lab, 'tail_excess']), fontsize=9)
    ax.set_xlabel('t [h]'); ax.grid(alpha=0.3); ax.legend(fontsize=7)
fig.suptitle('Fig 16 - obs vs pred con la cola de predicción sombreada (terminación = 5% del peak)', y=1.0)
fig.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.4))
for lab in LABS:
    axes[0].plot(AUD.loc[lab, 'int_pred'], AUD.loc[lab, 'int_obs'], 'o', ms=9, color=colors[lab])
    axes[0].annotate(lab, (AUD.loc[lab, 'int_pred'], AUD.loc[lab, 'int_obs']), textcoords='offset points', xytext=(6, 4), fontsize=8)
    axes[1].plot(range(len(LABS)), TAIL.loc[LABS, 'tail_excess'], 'o', ms=8, color=colors[lab])
    axes[1].annotate(lab, (LABS.index(lab), TAIL.loc[lab, 'tail_excess']), textcoords='offset points', xytext=(5, 4), fontsize=8)
lim = [0, max(AUD.int_pred.max(), AUD.int_obs.max()) * 1.08]
axes[0].plot(lim, lim, 'k--', lw=1)
axes[0].set_xlim(lim); axes[0].set_ylim(lim)
axes[0].set_xlabel('int_pred [g/L]'); axes[0].set_ylabel('int_obs [g/L]')
axes[0].set_title('Fig 17 - integral obs vs pred', fontsize=10)
axes[1].axhline(0, color='k', lw=1)
axes[1].set_xticks(range(len(LABS))); axes[1].set_xticklabels(LABS, rotation=60, fontsize=8)
axes[1].set_ylabel('tail_excess = post_peak_pred - post_peak_obs [g/L]')
axes[1].set_title('Fig 18 - exceso de cola predicha por LAB', fontsize=10)
axes[0].grid(alpha=0.3); axes[1].grid(alpha=0.3)
fig.tight_layout(); plt.show()
print('Lectura: LAB011/012 conservan más área observada post-peak que la predicción full-theta;')
print('la intensidad exacta se informa en la tabla y se compara con el modelo anterior al final.')


## 16. Descomposición de la predicción: `qprod_base -> qprod_efectiva -> qgas -> qpred`

Se reutiliza el módulo del runner **en modo solo-lectura** (`build_driver_cache`, `raw_qgas_grid_prediction`),
pero su theta natural por defecto se reemplaza local y explícitamente por los 17 valores de `theta_natural_full.csv`. La capa CO2 proviene del experimento full-theta SCCM corregido; no se ejecuta fitting. La reconstrucción se
verifica contra `prediction_rows.csv` (debe reproducirse exactamente). Capas:
- `qprod_base`: producción biológica upstream (θ_natural, T e ICs del batch).
- `qprod_O2` = efectiva **sin gate** (pool O2 inicial + fracción fermentativa; `chemistry_aligned=False`).
- `qprod_efectiva`: con gate químico de activación + respuesta al pulso de N (rampa/gain congelados).
- `qgas`: pool disuelto + release continuo (k, Csat congelados). `qpred = matrix_gain * qgas`.

ADVERTENCIA (repetida): `ratio_int < 1` NO es evidencia de microfuga; esta sección localiza DÓNDE aparece cada discrepancia.

In [ ]:
import sys as _sys
_sys.path.insert(0, FM)
from laboratory_2026 import run_co2_matrix_cross_validation_2026 as co2x
from dataclasses import replace as _replace

batches_all, theta_by_matrix, _nat = co2x.load_batches()
theta_by_matrix['natural'] = dict(zip(theta_full_table['parameter'], theta_full_table['theta'].astype(float)))
assert len(theta_by_matrix['natural']) == 17
assert set(theta_by_matrix['natural']) == expected_theta_parameters
_fp = pd.read_csv(os.path.join(RES, 'fit_parameters.csv'))
PN = dict(zip(_fp[_fp.calibration_matrix == 'natural'].parameter, _fp[_fp.calibration_matrix == 'natural'].estimate))
assert set(PN) == set(manifest['co2_parameters'])
display(_fp[_fp.calibration_matrix == 'natural'][['parameter', 'estimate', 'lower_bound', 'upper_bound', 'active_bound']].reset_index(drop=True))
cache, _diag = co2x.build_driver_cache(batches_all, theta_by_matrix, obs)

def qgas_layers(c, chem_align=True):
    qg, qe, _o2, _phi = co2x.raw_qgas_grid_prediction(
        c, PN['kCO2_release_h'], PN['CO2sat_scale'], PN['O2_qmax_mg_gdw_h'], PN['O2_initial_scale'],
        chemistry_aligned=chem_align, nitrogen_boost_transition=True,
        pulse_t_rise_h=PN['pulse_t_rise_h'], pulse_activity_gain=PN['pulse_activity_gain'],
        continuous_release=True, bounded_chemical_activation=chem_align,
        chem_activation_start_fraction=PN['chem_activation_start_fraction'],
        chem_activation_duration_fraction=PN['chem_activation_duration_fraction'])
    return qe, qg

drows = []
for lab in LABS:
    c = cache[lab]
    t = c.time_h
    eff, qgas = qgas_layers(c, True)
    eff_ng, _ = qgas_layers(c, False)
    g = prn[prn.batch == lab].sort_values('time_h')
    pred_chk = PN['matrix_gain'] * np.interp(g.time_h.values, t, qgas)
    dmax = float(np.max(np.abs(pred_chk - np.nan_to_num(g.predicted_g_l_h.values))))
    drows.append(dict(LAB=lab, horizon=float(t[-1]),
        int_qprod_base=float(np.trapz(c.base_qprod_g_l_h, t)),
        int_qprod_O2_sin_gate=float(np.trapz(eff_ng, t)),
        int_qprod_efectiva=float(np.trapz(eff, t)),
        int_qgas=float(np.trapz(qgas, t)),
        int_qpred=float(PN['matrix_gain'] * np.trapz(qgas, t)),
        int_qobs=float(AUD.loc[lab, 'int_obs']),
        gate_cut_pct=100 * (1 - np.trapz(eff, t) / np.trapz(eff_ng, t)),
        verif_maxdiff_vs_predrows=dmax))
DEC = pd.DataFrame(drows).set_index('LAB')
DEC.to_csv(os.path.join(OUT, 'prediction_decomposition_by_lab.csv'))
display(DEC.round(2))
print('Verificación de reconstrucción: max |dif| vs prediction_rows.csv =', float(DEC.verif_maxdiff_vs_predrows.max()))
print('(coincidencia exacta a precisión de máquina -> la descomposición reproduce el artefacto canónico).')

In [ ]:
fig, ax = plt.subplots(figsize=(10.5, 5))
x = np.arange(len(LABS))
w = 0.13
series = [('int_qprod_base', 'qprod_base (upstream)'), ('int_qprod_O2_sin_gate', 'qprod_O2 (sin gate)'),
          ('int_qprod_efectiva', 'qprod_efectiva (gate+pulso)'), ('int_qgas', 'qgas (pool/release)'),
          ('int_qpred', 'qpred = matrix_gain*qgas')]
for k, (col, lab_) in enumerate(series):
    ax.bar(x + (k - 2) * w, DEC.loc[LABS, col], width=w, label=lab_)
ax.plot(x, DEC.loc[LABS, 'int_qobs'], 'kD', ms=9, label='int_qobs (sensor)')
ax.set_xticks(x); ax.set_xticklabels(LABS, rotation=60, fontsize=9)
ax.set_ylabel('integral [g/L] (grilla del driver)')
ax.set_title('Fig 19 - Descomposición de la cadena de predicción vs CO2 observado')
ax.legend(fontsize=8); ax.grid(alpha=0.3, axis='y')
fig.tight_layout(); plt.show()
print('RESULTADO DEL CÁLCULO FULL-THETA:')
print('Rango integral upstream qprod_base: %.2f-%.2f g/L.' % (DEC.int_qprod_base.min(), DEC.int_qprod_base.max()))
for lab in LABS:
    print('%s: gate_cut %.1f%% | int_qpred %.2f | int_qobs %.2f g/L' % (
        lab, DEC.loc[lab, 'gate_cut_pct'], DEC.loc[lab, 'int_qpred'], DEC.loc[lab, 'int_qobs']))
print('La descomposición localiza la discrepancia dentro de la referencia full-theta; no identifica por sí sola una pérdida física.')

## 17. Contrafactuales con el modelo congelado (forward puro, sin fitting)

Para LAB010: ¿su predicción baja se debe a T, a X0/ICs o al gate? Se re-simula intercambiando inputs
(perfil T, X0, ICs completas de otro batch). Para LAB011/012: ¿su exceso observado se reproduce con
X0/T de otros batches? Si ningún swap lo reproduce, la causa queda **no identificada** (limitación
estructural del modelo o factor no modelado) y NO se atribuye a biología/ICs sin evidencia.

In [ ]:
def pred_metrics_cf(batch):
    sub = obs[obs.batch == batch.batch]
    c2, _ = co2x.build_driver_cache({batch.batch: batch}, theta_by_matrix, sub)
    c = c2[batch.batch]
    _qe, qg = qgas_layers(c, True)
    q = PN['matrix_gain'] * qg
    return float(q.max()), float(np.trapz(q, c.time_h))

b010, b011, b012, b004, b008 = batches_all['LAB010'], batches_all['LAB011'], batches_all['LAB012'], batches_all['LAB004'], batches_all['LAB008']
cf_rows = []
def add(nombre, batch, nota):
    pk, it = pred_metrics_cf(batch)
    cf_rows.append(dict(contrafactual=nombre, qpred_peak=pk, qpred_integral=it, nota=nota))
add('LAB010 base', b010, 'referencia')
add('LAB010 + T(LAB011: 18-15-21)', _replace(b010, temperature_c=np.interp(b010.time, b011.time, b011.temperature_c)), 'swap T')
add('LAB010 + X0(LAB004: 0.394)', _replace(b010, initials={**b010.initials, 'X': b004.initials['X']}), 'swap X0')
add('LAB010 + ICs completas(LAB004)', _replace(b010, initials=dict(b004.initials)), 'swap ICs')
add('LAB011 base', b011, 'referencia (obs: peak 0.877, int 35.7)')
add('LAB011 + X0(LAB008: 0.253)', _replace(b011, initials={**b011.initials, 'X': b008.initials['X']}), 'swap X0')
add('LAB011 + T(LAB008: A)', _replace(b011, temperature_c=np.interp(b011.time, b008.time, b008.temperature_c)), 'swap T')
add('LAB011 + T y X0(LAB008)', _replace(b011, temperature_c=np.interp(b011.time, b008.time, b008.temperature_c), initials={**b011.initials, 'X': b008.initials['X']}), 'swap T+X0')
add('LAB012 base', b012, 'referencia (obs: peak 0.835, int 41.8)')
add('LAB012 + X0(LAB008)', _replace(b012, initials={**b012.initials, 'X': b008.initials['X']}), 'swap X0')
CF = pd.DataFrame(cf_rows)
CF.to_csv(os.path.join(OUT, 'counterfactuals_frozen_model.csv'), index=False)
display(CF.round(3))
for target in ['LAB010', 'LAB011', 'LAB012']:
    subset = CF[CF.contrafactual.str.startswith(target)]
    print('%s contrafactuales: peak %.3f-%.3f | integral %.2f-%.2f g/L' % (
        target, subset.qpred_peak.min(), subset.qpred_peak.max(), subset.qpred_integral.min(), subset.qpred_integral.max()))
print('Lectura: los swaps conservan exactamente los mecanismos originales y ahora usan theta natural full + capa CO2 recalibrada.')
print('Si no alcanzan la observación, la discrepancia permanece no explicada dentro de esta estructura; no se atribuye automáticamente a biología ni a fuga.')

## 18. LAB016–LAB018 como referencia independiente de reproducibilidad (pre-nutrición)

Mismo mosto, ~20 °C isotermo, t0 físico medido y mejor estanqueidad. **NO son datos de calibración** y su
notebook holdout no se toca. Sin química G/F/E/YAN durante la fermentación: NO hay balance de carbono
equivalente al de §6 (no se inventa). Cadena de proceso (declarada): `CO2_FILT` alineado a t0 ->
mediana móvil 61 pts -> corrección de cero (cuantil 10% de las primeras 6 h pre-onset) -> conversión
SCCM corregida -> bin horario -> onset sostenido (baseline + 10% del peak pre-nutrición, 3 puntos).
Ventana primaria PRE-NUTRICIÓN: onset -> +56.93 h desde t0 (primer pulso tardío). El registro completo
se muestra solo como contexto (LAB017 tiene un segundo pulso distinto).

In [ ]:
NUT_H = 56.93
res16, series16 = {}, {}
for lab in ['LAB016', 'LAB017', 'LAB018']:
    d = pd.read_csv(os.path.join(FM, 'data', 'mem2026', 'LAB016-018', '%s_co2_inoculation_aligned.csv' % lab))
    t = d.time_h.values
    f = d.flow_filt_sccm.rolling(61, center=True, min_periods=20).median().ffill().bfill()
    zero = float(f[t < 6].quantile(0.10))
    q = (f - zero).clip(lower=0) * SCCM_CORRECTED
    bins = np.arange(np.floor(t.min()), np.ceil(t.max()) + 1)
    idx = np.clip(np.digitize(t, bins) - 1, 0, len(bins) - 2)
    qh = pd.Series(q.values).groupby(idx).mean()
    th = 0.5 * (bins[:-1] + bins[1:])
    pre = th < NUT_H
    pk = float(qh.values[pre].max())
    base = float(qh.values[th < 6].mean())
    thr = max(0.05, base + 0.10 * (pk - base))
    okv = qh.values > thr
    onset = float('nan')
    for i in range(len(okv) - 2):
        if okv[i] and okv[i + 1] and okv[i + 2]:
            onset = float(th[i]); break
    m_pre = (th >= onset) & pre
    integ_pre = float(np.trapz(qh.values[m_pre], th[m_pre]))
    tpk = float(th[int(np.argmax(np.where(pre, qh.values, -1)))])
    dtf = pd.read_csv(os.path.join(FM, 'data', 'mem2026', 'LAB016-018', '%s_temperature_inoculation_aligned.csv' % lab))
    mT = (dtf.time_h.values >= onset) & (dtf.time_h.values < NUT_H)
    Tv = dtf['T'].values[mT]
    res16[lab] = dict(onset_h=onset, T_mean=float(np.mean(Tv)), T_med=float(np.median(Tv)),
        qmax_preN=pk, tpeak_rel=tpk - onset, integral_preN=integ_pre, width_eff=integ_pre / pk,
        subLOD_preN_pct=100 * float((qh.values[m_pre] < 0.05).mean()),
        integral_full=float(np.trapz(qh.values, th)), zero_offset_sccm=zero)
    series16[lab] = (th, qh.values, onset)
V16 = pd.DataFrame(res16).T
V16.to_csv(os.path.join(OUT, 'lab016_018_prenutrition_reference.csv'))
display(V16.astype(float).round(3))
for c in ['qmax_preN', 'integral_preN', 'tpeak_rel', 'width_eff']:
    V16[c] = pd.to_numeric(V16[c])
    V16['ratio_' + c] = V16[c] / V16[c].median()
print('CV(qmax_preN)=%.1f%% | CV(integral_preN)=%.1f%% | CV(tpeak_rel)=%.1f%%' % (
    100 * V16.qmax_preN.std() / V16.qmax_preN.mean(),
    100 * V16.integral_preN.std() / V16.integral_preN.mean(),
    100 * V16.tpeak_rel.std() / V16.tpeak_rel.mean()))
print('max/min: qmax %.2f | integral %.2f' % (V16.qmax_preN.max() / V16.qmax_preN.min(),
      V16.integral_preN.max() / V16.integral_preN.min()))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12.5, 8))
c16 = dict(zip(['LAB016', 'LAB017', 'LAB018'], plt.cm.Set1([0, 1, 2])))
for lab in ['LAB016', 'LAB017', 'LAB018']:
    th, q, onset = series16[lab]
    axes[0, 0].plot(th, q, lw=1.1, color=c16[lab], label=lab)
    axes[0, 1].plot(th - onset, q, lw=1.1, color=c16[lab], label=lab)
    axes[0, 2 - 1].axvline(NUT_H - onset if False else 0, color='grey', lw=0.5)
for ax, ttl, xlab in [(axes[0, 0], 'Fig 20 - pre-nutrición (tiempo desde t0)', 't desde t0 [h]'),
                      (axes[0, 1], 'Fig 21 - alineadas por onset', 'tau = t - onset [h]')]:
    ax.axvspan(NUT_H if 't0' in ttl else NUT_H - 0, NUT_H if False else 1e9, color='grey', alpha=0.0)
    ax.set_xlabel(xlab); ax.set_ylabel('qCO2 [g/L/h]'); ax.set_title(ttl, fontsize=10)
    ax.legend(fontsize=8); ax.grid(alpha=0.3)
axes[0, 0].axvline(NUT_H, color='k', ls=':', lw=1.2)
axes[0, 1].axvline(NUT_H - float(np.nanmean([series16[l][2] for l in series16])), color='k', ls=':', lw=1.2, label='nutrición (media)')
for lab in ['LAB016', 'LAB017', 'LAB018']:
    th, q, onset = series16[lab]
    pre = th < NUT_H
    axes[1, 0].plot(th[pre] - onset, q[pre] / q[pre].max(), lw=1.1, color=c16[lab], label=lab)
    dtf = pd.read_csv(os.path.join(FM, 'data', 'mem2026', 'LAB016-018', '%s_temperature_inoculation_aligned.csv' % lab))
    axes[1, 1].plot(dtf.time_h.values, dtf['T'].values, lw=0.9, color=c16[lab], label=lab)
axes[1, 0].set_xlabel('tau [h]'); axes[1, 0].set_ylabel('q / qmax')
axes[1, 0].set_title('Fig 22 - normalizadas por peak (pre-nutrición)', fontsize=10)
axes[1, 1].set_xlabel('t desde t0 [h]'); axes[1, 1].set_ylabel('T [C]')
axes[1, 1].set_title('Fig 23 - temperatura medida (SP 20 C)', fontsize=10)
for ax in axes[1]:
    ax.legend(fontsize=8); ax.grid(alpha=0.3)
fig.tight_layout(); plt.show()
import itertools as _it
grid = np.arange(0, 34, 0.5)
curves = {}
for lab in ['LAB016', 'LAB017', 'LAB018']:
    th, q, onset = series16[lab]
    m = (th - onset >= 0) & (th < NUT_H)
    tau = (th - onset)[m]
    curves[lab] = np.interp(grid, tau, q[m] / q[m].max())
for a, b in _it.combinations(curves, 2):
    print('corr forma %s vs %s: %.3f' % (a, b, float(np.corrcoef(curves[a], curves[b])[0, 1])))

## 19. Dispersión: históricos (LAB004–012) vs referencia LAB016–018

No se comparan magnitudes absolutas (T y condiciones distintas): se comparan **CV, ratio a mediana y
forma**. En históricos, el grupo térmico replicado (LAB004/005/008) es el análogo más directo del
triplicado de septiembre.

In [ ]:
grupoA = ['LAB004', 'LAB005', 'LAB008']
warm = ['LAB004', 'LAB005', 'LAB007', 'LAB008']
comp_rows = []
def cv(x):
    x = np.asarray(x, float)
    return 100 * x.std(ddof=1) / x.mean()
comp_rows.append(dict(conjunto='LAB016-018 pre-N (referencia, mejor sellado)', n=3, CV_qmax=cv(V16.qmax_preN),
                      CV_integr=cv(V16.integral_preN), rango_ratio_int=float(V16.ratio_integral_preN.max() - V16.ratio_integral_preN.min()),
                      nota='mismo mosto, 20 C, t0 fisico'))
comp_rows.append(dict(conjunto='grupo A historico (LAB004/005/008)', n=3, CV_qmax=cv(M.loc[grupoA, 'qmax']),
                      CV_integr=cv(M.loc[grupoA, 'integral']), rango_ratio_int=float((M.loc[grupoA, 'integral'] / M.loc[grupoA, 'integral'].median()).max() - (M.loc[grupoA, 'integral'] / M.loc[grupoA, 'integral'].median()).min()),
                      nota='programa A, abril, dos fechas'))
comp_rows.append(dict(conjunto='historicos calientes (004/005/007/008)', n=4, CV_qmax=cv(M.loc[warm, 'qmax']),
                      CV_integr=cv(M.loc[warm, 'integral']), rango_ratio_int=float((M.loc[warm, 'R_sugar'] / M.loc[warm, 'R_sugar'].median()).max() - (M.loc[warm, 'R_sugar'] / M.loc[warm, 'R_sugar'].median()).min()),
                      nota='sampling-repaired warm group'))
comp_rows.append(dict(conjunto='todos los historicos aceptados', n=8, CV_qmax=cv(M.qmax),
                      CV_integr=cv(M.integral), rango_ratio_int=float(M.R_rel.max() - M.R_rel.min()),
                      nota='programas mixtos + LAB010'))
DISP = pd.DataFrame(comp_rows)
display(DISP.round(2))
fig, axes = plt.subplots(1, 2, figsize=(12, 4.4))
axes[0].barh(DISP.conjunto, DISP.CV_qmax, color='#4c72b0', label='CV qmax')
axes[0].barh(DISP.conjunto, DISP.CV_integr, color='#dd8452', alpha=0.7, label='CV integral')
axes[0].set_xlabel('CV [%]'); axes[0].legend(fontsize=8); axes[0].set_title('Dispersión por conjunto', fontsize=10)
axes[0].grid(alpha=0.3, axis='x')
for lab in LABS:
    axes[1].plot(LABS.index(lab), M.loc[lab, 'R_rel'], 'o', ms=9, color=colors[lab])
    axes[1].annotate(lab, (LABS.index(lab), M.loc[lab, 'R_rel']), textcoords='offset points', xytext=(5, 4), fontsize=8)
lo = float(V16.ratio_integral_preN.min()); hi = float(V16.ratio_integral_preN.max())
axes[1].axhspan(lo, hi, color='green', alpha=0.18, label='banda LAB016-018 pre-N (ref)')
axes[1].axhline(1.0, color='k', lw=1)
axes[1].set_xticks(range(len(LABS))); axes[1].set_xticklabels(LABS, rotation=60, fontsize=8)
axes[1].set_ylabel('R_rel (históricos)'); axes[1].legend(fontsize=8)
axes[1].set_title('Fig 27 - recuperación relativa histórica vs banda de referencia', fontsize=10)
axes[1].grid(alpha=0.3)
fig.tight_layout(); plt.show()
print('HECHO NUMERICO: CV integral grupo A %.2f%%, warm %.2f%%, referencia %.2f%% y total %.2f%%.' % (
    DISP.loc[1, 'CV_integr'], DISP.loc[2, 'CV_integr'], DISP.loc[0, 'CV_integr'], DISP.loc[3, 'CV_integr']))
print('INTERPRETACION: los grupos calidos siguen comparables con la referencia, pero su dispersion aumenta')
print('respecto de la auditoria original. La dispersion total permanece dominada por LAB010 y LAB006.')


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13.2, 4.2))
labs16 = ['LAB016', 'LAB017', 'LAB018']
axes[0].bar(labs16, V16.qmax_preN, color=[c16[l] for l in labs16])
axes[0].set_ylabel('qmax pre-N [g/L/h]'); axes[0].set_title('Fig 24 - qmax pre-nutrición', fontsize=10)
axes[1].bar(labs16, V16.integral_preN, color=[c16[l] for l in labs16])
axes[1].set_ylabel('integral pre-N [g/L]'); axes[1].set_title('Fig 25 - integral pre-nutrición', fontsize=10)
axes[2].bar(labs16, V16.ratio_qmax_preN, color=[c16[l] for l in labs16], label='qmax/mediana')
axes[2].bar(labs16, V16.ratio_integral_preN, color=[c16[l] for l in labs16], alpha=0.5, label='integral/mediana')
axes[2].axhline(1.0, color='k', lw=1); axes[2].legend(fontsize=8)
axes[2].set_title('Fig 26 - desviación respecto de la mediana', fontsize=10)
for ax in axes:
    ax.grid(alpha=0.3, axis='y')
fig.tight_layout(); plt.show()
print('Registro completo (contexto): integrales 47.5 / 52.9 / 48.6 g/L con nutrición tardía incluida;')
print('LAB017 (doble pulso) termina con mas gas total: no se interpreta post-nutrición como fuga.')

## 20. Clasificación final sampling-repaired

Se conservan los criterios originales. Las señales observadas no cambian respecto de la versión sampling-repaired, pero toda evidencia dependiente del modelo se recalcula con theta natural full y la capa CO2 compatible. Las interpretaciones causales continúan condicionadas por las limitaciones experimentales y estructurales ya declaradas.


In [ ]:
cls2 = {}
for lab in LABS:
    rr = float(M.loc[lab, 'R_rel'])
    if lab == 'LAB006':
        cls2[lab] = dict(
            evidencia_termica='alta (16 C, ancho %.0f h)' % M.loc[lab, 'width_eff'],
            evidencia_modelo='cola predicha en exceso (%+.1f g/L; tail_share %.2f)' % (TAIL.loc[lab, 'tail_excess'], TAIL.loc[lab, 'tail_share_of_deficit']),
            evidencia_quimica='R_rel %.2f (moderadamente bajo)' % rr,
            recuperacion_gaseosa='levemente baja, no concluyente', confianza='media',
            conclusion='mixto: cinetica fria real + artefacto de cola del modelo; evidencia de perdida de gas INSUFICIENTE')
    elif lab == 'LAB010':
        cls2[lab] = dict(
            evidencia_termica='alta (15 C iso)',
            evidencia_modelo='prediccion baja = artefacto del gate (cut %.0f%%); no corrobora' % DEC.loc[lab, 'gate_cut_pct'],
            evidencia_quimica='R_rel %.2f con dGF/dE comparables' % rr,
            recuperacion_gaseosa='anormalmente baja', confianza='media-alta',
            conclusion='recuperacion gaseosa anormalmente baja: posible perdida de gas o lectura instrumental baja (F1, mayo); fermentador vs sensor no separable')
    elif lab in ('LAB011', 'LAB012'):
        cls2[lab] = dict(
            evidencia_termica='media (final calido)',
            evidencia_modelo='ratio_peak %.2f / ratio_int %.2f; exceso no reproducible con swaps X0/T' % (R.loc[lab, 'ratio_peak'], R.loc[lab, 'ratio_int']),
            evidencia_quimica='R_rel %.2f (alta, no deficitaria)' % rr,
            recuperacion_gaseosa='alta', confianza='media',
            conclusion='actividad gaseosa mayor que la predicha por el modelo; causa no identificada (limitacion estructural); no compatible con perdida')
    else:
        cls2[lab] = dict(
            evidencia_termica='parcial (programa calido)',
            evidencia_modelo='ratio_peak %.2f / ratio_int %.2f' % (R.loc[lab, 'ratio_peak'], R.loc[lab, 'ratio_int']),
            evidencia_quimica='R_rel %.2f' % rr,
            recuperacion_gaseosa='normal', confianza='alta',
            conclusion='compatible con cinetica normal; replica A consistente')
C2 = pd.DataFrame(cls2).T
C2.to_csv(os.path.join(OUT, 'final_classification_by_lab.csv'))
display(C2)
print('HECHO NUMERICO: clasificación recalculada con las mismas reglas y la referencia full-theta.')
print('INTERPRETACION: ninguna categoría se convierte por sí sola en evidencia demostrativa de fuga.')


## 21. Respuestas revisadas a las preguntas finales

In [ ]:
B = []
B.append(('1) ¿Qué referencia usa ahora la auditoría?',
          'Theta natural completo (%d parámetros), parámetros CO2 de la recalibración full-theta y SCCM_CORRECTED %.15f.' % (len(theta_full_table), SCCM_CORRECTED)))
B.append(('2) ¿Qué cambió en los datos observados?',
          'Nada respecto de sampling-repaired: LAB004/007/008 siguen reparados y los demás LAB conservan exactamente su señal previa.'))
B.append(('3) ¿Qué cambia para LAB011?',
          'Con full-theta: ratio_peak %.3f, ratio_int %.3f, RMSE %.3f g/L/h y peaks obs/pred %.3f/%.3f.' % (R.loc['LAB011','ratio_peak'], R.loc['LAB011','ratio_int'], R.loc['LAB011','rmse_g_l_h'], R.loc['LAB011','peak_obs'], R.loc['LAB011','peak_pred'])))
B.append(('4) ¿Qué cambia para LAB012?',
          'Con full-theta: ratio_peak %.3f, ratio_int %.3f, RMSE %.3f g/L/h y peaks obs/pred %.3f/%.3f.' % (R.loc['LAB012','ratio_peak'], R.loc['LAB012','ratio_int'], R.loc['LAB012','rmse_g_l_h'], R.loc['LAB012','peak_obs'], R.loc['LAB012','peak_pred'])))
B.append(('5) ¿Cambian las recuperaciones relativas observadas?',
          'No: R_sugar, R_rel e integrales observadas dependen sólo de las señales, que permanecen iguales a la auditoría sampling-repaired.'))
B.append(('6) ¿Cambia la evidencia de microfugas?',
          'La referencia predictiva cambia, pero una discrepancia modelo-observación no demuestra fuga. LAB010 conserva R_rel %.2f y fuga vs lectura baja sigue no separable.' % M.loc['LAB010', 'R_rel']))
B.append(('7) ¿Qué sigue robusto?',
          'La jerarquía de recuperación observada, la señal reparada, la referencia LAB016-018 y las limitaciones de identificación permanecen intactas.'))
for question, answer in B:
    print(question)
    print('   ->', answer)
    print()


## Cierre sampling-repaired + referencia natural full-theta

Artefactos actualizados en `laboratory_2026/results/co2_historical_variability_microleaks_2026_sampling_repaired/`: observaciones horarias reparadas, trazabilidad de theta/CO2/SCCM, métricas observado-predicho full-theta, clasificación, tail, descomposición, contrafactuales, referencia LAB016-LAB018 y comparaciones contra la referencia predictiva anterior.

**HECHO NUMÉRICO:** las integrales observadas y su variabilidad permanecen iguales a la auditoría sampling-repaired. Sólo cambian las métricas que incorporan la nueva predicción congelada.

**INTERPRETACIÓN:** la comparación full-theta corrige la referencia del modelo, pero no añade un mecanismo causal nuevo. La evidencia de microfuga continúa condicionada por las señales observadas, la química y la imposibilidad de separar reactor/canal/sensor.


## 22. Comparaciones de control: señal original/reparada y modelo anterior/full-theta

Primero se conserva la comparación de señal original frente a sampling-repaired bajo la referencia activa. Luego se compara, sobre exactamente la misma señal observada sampling-repaired, la predicción anterior contra full-theta + CO2 recalibrado. Se separan hechos numéricos de interpretación; una discrepancia de modelo no demuestra fuga.


In [ ]:
M_OLD = pd.read_csv(os.path.join(OUT_ORIGINAL, 'master_table_by_batch.csv'), index_col=0)
TAIL_OLD = pd.read_csv(os.path.join(OUT_ORIGINAL, 'tail_decomposition_by_lab.csv'), index_col=0)
C2_OLD = pd.read_csv(os.path.join(OUT_ORIGINAL, 'final_classification_by_lab.csv'), index_col=0)

comparison_affected_rows = []
old_integral_rank = M_OLD['integral'].rank(method='min', ascending=True).astype(int)
new_integral_rank = M['integral'].rank(method='min', ascending=True).astype(int)
old_rrel_rank = M_OLD['R_rel'].rank(method='min', ascending=True).astype(int)
new_rrel_rank = M['R_rel'].rank(method='min', ascending=True).astype(int)
for lab in REPAIRED_LABS:
    old_integral = float(M_OLD.loc[lab, 'integral'])
    new_integral = float(M.loc[lab, 'integral'])
    old_peak = float(M_OLD.loc[lab, 'qmax'])
    new_peak = float(M.loc[lab, 'qmax'])
    comparison_affected_rows.append({
        'LAB': lab,
        'integral_old_g_l': old_integral,
        'integral_new_g_l': new_integral,
        'integral_delta_g_l': new_integral - old_integral,
        'integral_change_pct': 100.0 * (new_integral - old_integral) / old_integral,
        'R_sugar_old': float(M_OLD.loc[lab, 'R_sugar']),
        'R_sugar_new': float(M.loc[lab, 'R_sugar']),
        'R_rel_old': float(M_OLD.loc[lab, 'R_rel']),
        'R_rel_new': float(M.loc[lab, 'R_rel']),
        'R_rel_delta': float(M.loc[lab, 'R_rel'] - M_OLD.loc[lab, 'R_rel']),
        'obs_pred_integral_old': old_integral / float(AUD.loc[lab, 'int_pred']),
        'obs_pred_integral_new': float(AUD.loc[lab, 'ratio_int']),
        'peak_old_g_l_h': old_peak,
        'peak_new_g_l_h': new_peak,
        'peak_change_pct': 100.0 * (new_peak - old_peak) / old_peak,
        't_peak_old_h': float(M_OLD.loc[lab, 't_peak']),
        't_peak_new_h': float(M.loc[lab, 't_peak']),
        'onset_old_h': float(M_OLD.loc[lab, 'onset']),
        'onset_new_h': float(M.loc[lab, 'onset']),
        'integral_rank_low_old': int(old_integral_rank.loc[lab]),
        'integral_rank_low_new': int(new_integral_rank.loc[lab]),
        'R_rel_rank_low_old': int(old_rrel_rank.loc[lab]),
        'R_rel_rank_low_new': int(new_rrel_rank.loc[lab]),
        'classification_old': str(C2_OLD.loc[lab, 'conclusion']),
        'classification_new': str(C2.loc[lab, 'conclusion']),
        'classification_changed': str(C2_OLD.loc[lab, 'conclusion']) != str(C2.loc[lab, 'conclusion']),
    })
OLD_NEW_AFFECTED = pd.DataFrame(comparison_affected_rows).set_index('LAB')
OLD_NEW_AFFECTED.to_csv(os.path.join(OUT, 'old_vs_sampling_repaired_affected_labs.csv'))
display(OLD_NEW_AFFECTED.round(4))

def cv_pct(values):
    values = np.asarray(values, dtype=float)
    return float(100.0 * values.std(ddof=1) / values.mean())
sets = {
    'group_A_004_005_008': ['LAB004', 'LAB005', 'LAB008'],
    'warm_004_005_007_008': ['LAB004', 'LAB005', 'LAB007', 'LAB008'],
    'all_accepted': LABS,
}
variability_rows = []
for label, labs in sets.items():
    variability_rows.append({
        'group': label,
        'CV_integral_old_pct': cv_pct(M_OLD.loc[labs, 'integral']),
        'CV_integral_new_pct': cv_pct(M.loc[labs, 'integral']),
        'CV_peak_old_pct': cv_pct(M_OLD.loc[labs, 'qmax']),
        'CV_peak_new_pct': cv_pct(M.loc[labs, 'qmax']),
        'R_rel_range_old': float(M_OLD.loc[labs, 'R_rel'].max() - M_OLD.loc[labs, 'R_rel'].min()),
        'R_rel_range_new': float(M.loc[labs, 'R_rel'].max() - M.loc[labs, 'R_rel'].min()),
    })
OLD_NEW_VARIABILITY = pd.DataFrame(variability_rows).set_index('group')
OLD_NEW_VARIABILITY.to_csv(os.path.join(OUT, 'historical_variability_old_vs_sampling_repaired.csv'))
display(OLD_NEW_VARIABILITY.round(3))

print('HECHOS NUMÉRICOS:')
for lab in REPAIRED_LABS:
    row = OLD_NEW_AFFECTED.loc[lab]
    print(
        f"- {lab}: integral {row.integral_old_g_l:.3f}->{row.integral_new_g_l:.3f} g/L "
        f"({row.integral_delta_g_l:+.3f}; {row.integral_change_pct:+.2f}%), "
        f"R_rel {row.R_rel_old:.3f}->{row.R_rel_new:.3f}, "
        f"obs/pred {row.obs_pred_integral_old:.3f}->{row.obs_pred_integral_new:.3f}."
    )
print('\nINTERPRETACIÓN:')
print('- LAB004 recupera area y aumenta su exceso observado frente a la prediccion; esto debilita una lectura de baja recuperacion asociada a sus depresiones, sin demostrar ausencia de fuga.')
print('- LAB007 recupera área moderada y conserva peak/timing; su clasificación normal se mantiene.')
print('- LAB008 conserva prácticamente la misma integral neta porque se eliminan caídas y rebotes; cambia el peak, no aparece evidencia nueva de pérdida gaseosa.')
print('- LAB010 y LAB006 siguen dominando las recuperaciones bajas; ninguna de sus señales fue alterada.')
print('- La conclusión general sobre microfugas se mantiene: solo LAB010 conserva evidencia compatible, no demostrativa, de pérdida/lectura baja; LAB006 continúa explicado por cinética fría + cola del modelo.')

def frozen_prediction_metrics(frame):
    rows = []
    for lab in LABS:
        g = frame[frame.batch == lab].sort_values('time_h')
        t = g.time_h.to_numpy(dtype=float)
        o = np.nan_to_num(g.observed_g_l_h.to_numpy(dtype=float), nan=0.0)
        p = np.nan_to_num(g.predicted_g_l_h.to_numpy(dtype=float), nan=0.0)
        valid = ~g.left_censored.astype(bool).to_numpy()
        rows.append({
            'LAB': lab, 'ratio_peak': float(o.max() / p.max()),
            'ratio_int': float(np.trapz(o, t) / np.trapz(p, t)),
            'rmse_g_l_h': float(np.sqrt(np.mean((o[valid] - p[valid]) ** 2))),
            'integral_observed_g_l': float(np.trapz(o, t)),
            'integral_predicted_g_l': float(np.trapz(p, t)),
            'peak_observed_g_l_h': float(o.max()), 'peak_predicted_g_l_h': float(p.max()),
            't_peak_observed_h': float(t[o.argmax()]), 't_peak_predicted_h': float(t[p.argmax()]),
        })
    return pd.DataFrame(rows).set_index('LAB')

MODEL_OLD = frozen_prediction_metrics(prn_old_model)
MODEL_FULL = frozen_prediction_metrics(prn)
assert np.array_equal(MODEL_OLD['integral_observed_g_l'].to_numpy(), MODEL_FULL['integral_observed_g_l'].to_numpy())
model_comparison_rows = []
for lab in LABS:
    row = {'LAB': lab}
    for metric in MODEL_OLD.columns:
        row[metric + '_old_model'] = float(MODEL_OLD.loc[lab, metric])
        row[metric + '_full_theta'] = float(MODEL_FULL.loc[lab, metric])
    row['classification_old_model'] = str(C2_OLD.loc[lab, 'conclusion'])
    row['classification_full_theta'] = str(C2.loc[lab, 'conclusion'])
    row['classification_changed'] = row['classification_old_model'] != row['classification_full_theta']
    model_comparison_rows.append(row)
MODEL_REFERENCE_COMPARISON = pd.DataFrame(model_comparison_rows).set_index('LAB')
MODEL_REFERENCE_COMPARISON.to_csv(os.path.join(OUT, 'model_reference_old_vs_full_theta_by_lab.csv'))
display(MODEL_REFERENCE_COMPARISON.loc[['LAB011', 'LAB012']].T.round(5))
print('\nHECHO NUMÉRICO — LAB011/LAB012, modelo anterior -> full-theta:')
for lab in ['LAB011', 'LAB012']:
    old = MODEL_OLD.loc[lab]; new = MODEL_FULL.loc[lab]
    print('%s: ratio_peak %.3f->%.3f | ratio_int %.3f->%.3f | RMSE %.3f->%.3f | ' % (
        lab, old.ratio_peak, new.ratio_peak, old.ratio_int, new.ratio_int, old.rmse_g_l_h, new.rmse_g_l_h)
        + 'int_pred %.3f->%.3f g/L | peak_pred %.3f->%.3f g/L/h | t_peak_pred %.0f->%.0f h' % (
        old.integral_predicted_g_l, new.integral_predicted_g_l, old.peak_predicted_g_l_h, new.peak_predicted_g_l_h,
        old.t_peak_predicted_h, new.t_peak_predicted_h))
print('INTERPRETACIÓN:')
print('- LAB011: la discrepancia integral y el RMSE aumentan; el peak cambia poco y su timing mejora. La evidencia de limitación estructural se fortalece en área total.')
print('- LAB012: disminuye la discrepancia de amplitud de peak, pero aumentan ratio_int y RMSE y empeora el timing predicho. La evidencia estructural se mantiene y aumenta en área/timing.')
print('- En ambos, el exceso observado sigue siendo opuesto a la firma esperada de una microfuga.')
